# Lecture 02 Homework (Advanced): Teach the Course Glossary with LoRA and Compare It with Glossary Search

This notebook is the course notebook `lora_format_tuning.ipynb` adapted to the **Advanced** homework. Model, pinned libraries, model loading, `generate()`, LoRA settings, and the `SFTTrainer` setup are unchanged. What changes:

| Section | Change from the original notebook |
| --- | --- |
| 1 | `VARIANT` flag (`"plain"` / `"augmented"`) and `EPOCHS` (default 20 = original value) → `RUN_NAME` (e.g. `plain_ep20`) used in all output file names; `max_new_tokens` 96 → 150 |
| 2 | Training data = output of `scripts/make_sft_dataset.py` (embedded, sha256-checked) instead of the FAQ; own evaluation set (in-glossary / paraphrased / absent) |
| 3 | Questions are plain user messages (same format as training, no FAQ system prompt); new metrics: keyword hit, token F1, abstained vs asserted |
| 4 | Unchanged except the training data (no validation split); full `log_history` saved to `loss_{RUN_NAME}.json` |
| 5 | Per-group before/after comparison, regressions, side-by-side table |
| 6 | Replaced: **comparison with a deterministic glossary search** (same metrics, plus latency), and **search + model** (retrieve the entry, then generate from it) with the base and trained model |
| 7 | `result_{RUN_NAME}.json` + per-question eval files + loss log, download cell |
| 8 | 4-bit load of the base model vs 32-bit on the 35 questions (observation, no training); **on by default** (`RUN_4BIT = True`), runs before section 7 so it is saved and downloaded |

Everything needed (glossary, training JSONL, evaluation set) is embedded in the notebook, so **no file upload is needed**. Set `VARIANT` and `EPOCHS` in section 1, then run cells in order from the top on a Colab **T4 GPU**. Recommended: `plain` with `EPOCHS = 20` (300 steps, ≈ 11–12 min of training) and `augmented` with `EPOCHS = 14` (392 steps, ≈ 15 min). Total runtime ≈ 27–33 minutes per run including section 8 (model load + 4 evaluation passes of ≈ 1.5–4.5 min each + training).

**Question this notebook answers**: can a 0.8B model memorize a 58-term course glossary with LoRA, does it generalize to paraphrases and aliases, does it learn to say "not in the glossary", and how does that compare with simply searching the glossary?

## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

In [ ]:
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec}

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. Settings

**Changes from the original**: `VARIANT` selects the training data, `EPOCHS` sets `num_train_epochs` (default 20, the original value), and `max_new_tokens` is 150. `RUN_NAME = f"{VARIANT}_ep{EPOCHS}"` (e.g. `plain_ep20`) is used in every output file name, so runs with different settings never overwrite each other. `lr`, `lora_r`, `lora_alpha`, seed, model revision, and greedy decoding are unchanged.

- `VARIANT = "plain"`: exactly the output of `scripts/make_sft_dataset.py` (116 records = 58 terms × 2 templates). **This is the required baseline.**
- `VARIANT = "augmented"`: plain + alias questions ("What does {alias} mean?") + 16 abstention records for 8 training-only terms absent from the glossary ("That term is not in the course glossary."). Tests whether training can teach alias recall and "I don't know", and how far that generalizes to the evaluation questions (which never appear in training).
- To compare both, run the notebook twice (**Runtime → Restart session** between runs) and download both `result_{RUN_NAME}.json` files.
- **Epochs** (`EPOCHS`): plain has 116 records → 29 batches of 4 → **15 optimizer steps per epoch** (accumulation 2); augmented has 217 → 55 batches → **28 steps per epoch**. Measured on a T4 in an earlier 3-epoch run: ≈ 2.27 s per step for both (45 steps in 102 s, 84 steps in 191 s, including checkpoint saves). 3 epochs left the model undertrained (final loss ≈ 1.0, in-glossary keyword hit 0.08), so the default is the original notebook's 20.

  | Run | `EPOCHS` | Steps | Training time on a T4 (est.) | Passes per definition |
  | --- | --- | --- | --- | --- |
  | `plain` (required) | 20 | 300 | ≈ 11–12 min | 40 (2 templates × 20) |
  | `augmented` | 14 | 392 | ≈ 15 min | 28 + alias questions |

  More epochs memorize the 58 definitions more exactly; the risk is overfitting to the two question templates, which the `paraphrased` and `absent` groups measure.
- `max_new_tokens = 150`: definitions are 1–4 sentences (longest: 59 words with code identifiers ≈ 110 tokens), so 150 avoids truncating a correct answer.

**What to Look For**: the settings are displayed as JSON, including `variant`.

In [ ]:
from pathlib import Path
import os, json, time

VARIANT = "plain"   # "plain" = script output only (required baseline) | "augmented" = + alias questions + abstention examples
EPOCHS = 20         # Original lecture value. Recommended: plain 20 (300 steps, ~11-12 min on a T4) | augmented 14 (392 steps, ~15 min)
assert VARIANT in ("plain", "augmented") and isinstance(EPOCHS, int) and EPOCHS >= 1
RUN_NAME = f"{VARIANT}_ep{EPOCHS}"   # Used in every output file name, e.g. result_plain_ep20.json

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Small instruction-tuned model (Apache-2.0). Fits on a T4 without quantization
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Pinned to use the same weights on every run
    "variant": VARIANT,
    "run_name": RUN_NAME,
    "seed": 42,
    # 116 records (plain) / 217 (augmented) at 8 examples per step = 15 / 28 steps per epoch.
    # About 2.3 s per step on a T4: plain x 20 epochs = 300 steps (~11-12 min), augmented x 14 = 392 steps (~15 min)
    "num_train_epochs": EPOCHS,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA matrix dimension (smaller means fewer trainable parameters)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 150,   # Definitions are 1-4 sentences (longest about 110 tokens); 150 avoids truncation
    "gen": {"do_sample": False},   # Fix evaluation to greedy generation (consistent before/after conditions)
    "work_dir": "/content/qlora_lab",                          # Result file location (visible in the file list on the left)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Save location when USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Set True to mount Google Drive and save adapters and outputs
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. Glossary Training Data and Evaluation Set

**What to Do**: Load the embedded data, build the training set for the selected `VARIANT`, and check the evaluation set against it.

The first cell only holds data (it is long; you do not need to read it):

- `GLOSSARY`: `data/glossary.json` from the course repository (58 terms with `id`, `term`, `aliases`, `definition`, ...).
- `SFT_JSONL_LINES`: the exact output of `python3 scripts/make_sft_dataset.py > glossary_sft.jsonl` (116 lines, `messages` format). `SFT_SHA256` is the sha256 of that file; the second cell recomputes it and also regenerates the records from `GLOSSARY` with the script's templates to prove they match.
- `EVAL_SET`: my own evaluation set (also saved as `eval_set.json`). Expected answers are not passed to the model.

**Evaluation groups** (35 questions)

| Group | n | How the questions were made | Expected behavior |
| --- | --- | --- | --- |
| `in_glossary` | 12 | Exact training templates (`What is {term}?` / `Briefly explain {term}.`) for terms chosen to mix general AI terms (LLM, LoRA) and course-specific ones (Pi, default model, answer.md) | Glossary definition; measures memorization of **seen** questions |
| `paraphrased` | 13 | New wording, aliases, symptoms, a spelling variant; never in any training set (each has a `why` note) | Glossary definition of the expected term |
| `absent` | 10 | Common AI terms checked to be absent from glossary terms and aliases | Say it is not in the course glossary rather than inventing a course-specific definition |

`must_include`: 2–3 keywords per glossary question, chosen by hand from the gold definition to be **distinctive** (course-specific facts such as `gemini-3.5-flash-lite`, `Lecture 02`, `requires_approval`, or specific phrasing such as `expressive capacity`) so that a plausible general-knowledge answer is likely to miss them.

**What to Look For**: `sha256 OK`, `train=116` (plain) or `train=217` (augmented), `overlap=0`, and that no absent term matches the glossary.

In [ ]:
# Embedded data (generated from the course repository; do not edit by hand)
GLOSSARY = {'version': '2026-draft-1',
 'note': 'Draft glossary for Lecture 01–03 and Lecture 05 of AIE2026. Replace content once the lecture slides are finalized.',
 'terms': [{'id': 'ai-engineering',
            'term': 'AI engineering',
            'aliases': ['AI Engineering'],
            'lecture': 1,
            'category': 'Overview',
            'definition': 'The technical field of designing, building, evaluating, and operating useful applications and systems using '
                          'pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, connecting them to '
                          'data and tools, and iterating on evaluation and improvement, rather than building models from scratch.',
            'related': ['llm', 'evaluation-loop']},
           {'id': 'evaluation-loop',
            'term': 'Evaluation and improvement cycle',
            'aliases': ['improvement cycle'],
            'lecture': 1,
            'category': 'Overview',
            'definition': 'A development approach that repeats build → measure → improve. Since AI application outputs vary '
                          'probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, '
                          'data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a '
                          'learning support agent.',
            'related': ['ai-engineering', 'hallucination']},
           {'id': 'llm',
            'term': 'LLM (Large Language Model)',
            'aliases': ['large language model'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'A model trained on large amounts of text that generates text by predicting the tokens following its input. It '
                          'can handle many tasks, including question answering, summarization, and code generation, simply by changing the '
                          'instructions (prompt).',
            'related': ['token', 'prompt', 'pretraining']},
           {'id': 'token',
            'term': 'Token',
            'aliases': ['token'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API '
                          'charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; '
                          'for example, Japanese tends to require more tokens per character than English.',
            'related': ['context-window', 'rate-limit']},
           {'id': 'context-window',
            'term': 'Context window',
            'aliases': ['context length', 'context window'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and '
                          'tool results must also fit within this limit, making the choice of what to pass to the model important.',
            'related': ['token', 'agent-loop']},
           {'id': 'prompt',
            'term': 'Prompt',
            'aliases': ['prompt'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'Input text passed to an LLM. In addition to questions, it can include roles, constraints, output formats, and '
                          'examples to guide the output.',
            'related': ['system-prompt', 'llm']},
           {'id': 'system-prompt',
            'term': 'System prompt',
            'aliases': ['instruction text', 'instructions', 'system instruction'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': "Input that specifies model behavior (role, tone, rules, and so on) separately from the user's question. In "
                          'Pydantic AI, it is provided through the Agent instructions argument.',
            'related': ['prompt', 'guardrail']},
           {'id': 'temperature',
            'term': 'temperature',
            'aliases': ['temperature parameter'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher '
                          'values make them more varied. This is one reason the same question can yield different answers.',
            'related': ['llm', 'evaluation-loop']},
           {'id': 'hallucination',
            'term': 'Hallucination',
            'aliases': ['fabrication', 'hallucination'],
            'lecture': 1,
            'category': 'LLM basics',
            'definition': 'The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more '
                          'likely when asked about things the model does not know, such as recent information or organization- or '
                          'course-specific details. Mitigations include consulting external data through tools, asking for evidence, and '
                          'detecting errors through evaluation.',
            'related': ['tool', 'evaluation-loop']},
           {'id': 'api',
            'term': 'API',
            'aliases': ['Application Programming Interface', 'Web API'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'An interface through which a program calls the functionality of another service. In this course, we use an LLM '
                          'by calling the Gemini API from Python.',
            'related': ['api-key', 'rate-limit']},
           {'id': 'api-key',
            'term': 'API key',
            'aliases': ['API key'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage '
                          'allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as '
                          'a Codespaces secret and read it through the GEMINI_API_KEY environment variable.',
            'related': ['codespaces-secret', 'environment-variable']},
           {'id': 'environment-variable',
            'term': 'Environment variable',
            'aliases': ['environment variable', 'env'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'A value set outside a program (in the OS or execution environment) that the program can read by name. Used to '
                          'separate secrets such as API keys and environment-specific settings from code.',
            'related': ['api-key', 'codespaces-secret']},
           {'id': 'rate-limit',
            'term': 'Rate limit',
            'aliases': ['rate limit', '429 error'],
            'lecture': 1,
            'category': 'Development basics',
            'definition': 'A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free '
                          'tiers have particularly strict limits, and agents reach them easily because they call the LLM multiple times '
                          'for one question.',
            'related': ['api', 'agent-loop', 'default-model']},
           {'id': 'codespaces',
            'term': 'GitHub Codespaces',
            'aliases': ['Codespaces', 'Codespace'],
            'lecture': 1,
            'category': 'Development environment',
            'definition': 'A cloud development environment provided by GitHub. Open VS Code in a browser and immediately use an '
                          'environment configured by the repository. Personal accounts have a monthly free allowance; if you do not stop '
                          'the Codespace when idle, it continues consuming that allowance.',
            'related': ['devcontainer', 'codespaces-secret']},
           {'id': 'devcontainer',
            'term': 'devcontainer',
            'aliases': ['Dev Container', 'devcontainer.json'],
            'lecture': 1,
            'category': 'Development environment',
            'definition': 'A mechanism for defining a development environment as a container. Specify the Python version, extensions, and '
                          'startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment.',
            'related': ['codespaces']},
           {'id': 'codespaces-secret',
            'term': 'Codespaces secret',
            'aliases': ['Codespaces secrets'],
            'lecture': 1,
            'category': 'Development environment',
            'definition': 'A secret value registered in your GitHub account settings. It is passed as an environment variable when you '
                          'start Codespaces for a selected repository. When registering it, select the repository you will use (your own '
                          'fork). If registered while a Codespace is running, restart the Codespace.',
            'related': ['api-key', 'environment-variable', 'codespaces']},
           {'id': 'fork',
            'term': 'fork',
            'aliases': ['forking'],
            'lecture': 1,
            'category': 'Git/GitHub',
            'definition': "Copying someone else's entire repository to your own account. In this course, fork the course repository and "
                          'work and submit within your own fork.',
            'related': ['branch', 'pull-request']},
           {'id': 'branch',
            'term': 'Branch',
            'aliases': ['branch'],
            'lecture': 1,
            'category': 'Git/GitHub',
            'definition': 'A mechanism for separating streams of work. Try changes without breaking main, then merge them into main '
                          'through a PR once ready. In this course, create a working branch such as lec01 for each lecture.',
            'related': ['pull-request', 'fork']},
           {'id': 'pull-request',
            'term': 'Pull Request (PR)',
            'aliases': ['PR', 'Pull Request'],
            'lecture': 1,
            'category': 'Git/GitHub',
            'definition': 'A mechanism for requesting that changes from one branch be merged into another. It supports reviewing '
                          'differences, comments, and displaying automated test results. Submit work for this course as a PR from your '
                          'working branch to main within your own fork.',
            'related': ['branch', 'answer-md']},
           {'id': 'github-copilot',
            'term': 'GitHub Copilot',
            'aliases': ['Copilot'],
            'lecture': 1,
            'category': 'AI coding',
            'definition': "GitHub's AI coding assistance. It supports code completion in the editor, questions in chat, and editing "
                          'multiple files in Agent mode. Copilot Free limits completion counts and chat usage.',
            'related': ['code-completion', 'coding-agent']},
           {'id': 'code-completion',
            'term': 'Code completion',
            'aliases': ['inline completion', 'inline suggestion'],
            'lecture': 1,
            'category': 'AI coding',
            'definition': 'A feature in which AI suggests code to follow partially written code or comments. Accept suggestions with Tab '
                          'or a similar action. Suggestions may be incorrect, so read and understand them before accepting.',
            'related': ['github-copilot']},
           {'id': 'coding-agent',
            'term': 'Coding agent',
            'aliases': ['Coding Agent', 'AI coding agent'],
            'lecture': 1,
            'category': 'AI coding',
            'definition': 'AI that autonomously carries out requested programming work over multiple steps, using tools to read and write '
                          'files and run commands. Examples include Claude Code, Copilot Agent mode, and Pi. Internally, it has the same '
                          'LLM + loop + tools structure as an AI agent.',
            'related': ['ai-agent', 'harness']},
           {'id': 'pretraining',
            'term': 'Pretraining',
            'aliases': ['pre-training', 'pretraining'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'The initial stage of training a model to predict the next token using large amounts of general-purpose text. '
                          'Much of its language knowledge and general common sense is acquired here.',
            'related': ['fine-tuning', 'llm']},
           {'id': 'fine-tuning',
            'term': 'Fine-tuning',
            'aliases': ['fine-tuning', 'additional training', 'FT'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'Further training a pretrained model on data suited to a specific task or format. It is suitable for teaching '
                          'output format, tone, or behavior in a specific domain. For up-to-date facts, consulting external data through '
                          'tools or RAG is often more suitable.',
            'related': ['sft', 'peft', 'lora']},
           {'id': 'sft',
            'term': 'Supervised Fine-Tuning (SFT)',
            'aliases': ['SFT', 'Supervised Fine-Tuning', 'instruction tuning'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A method of further training a model using pairs of inputs and desired outputs, such as questions and model '
                          'answers. A representative method for teaching instruction-following behavior.',
            'related': ['fine-tuning', 'lora']},
           {'id': 'peft',
            'term': 'PEFT',
            'aliases': ['Parameter-Efficient Fine-Tuning'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'An umbrella term for methods that train only a small subset of model parameters or a small set of added '
                          'parameters instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a '
                          'representative example.',
            'related': ['lora', 'fine-tuning']},
           {'id': 'lora',
            'term': 'LoRA',
            'aliases': ['Low-Rank Adaptation'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices '
                          '(low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results '
                          '(adapters) in small files. The Lecture 02 exercises cover LoRA.',
            'related': ['peft', 'lora-rank', 'qlora']},
           {'id': 'lora-rank',
            'term': 'Rank (LoRA r)',
            'aliases': ['rank', 'r'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases '
                          'expressive capacity, but also increases the number of trainable parameters and memory use.',
            'related': ['lora']},
           {'id': 'qlora',
            'term': 'QLoRA',
            'aliases': ['Quantized LoRA'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. '
                          'This lets smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we '
                          'train with LoRA and compare memory and output differences when loading in 4-bit in the advanced section.',
            'related': ['lora', 'quantization']},
           {'id': 'quantization',
            'term': 'Quantization',
            'aliases': ['quantization'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce '
                          'memory use and computation. Accuracy may decrease slightly.',
            'related': ['qlora']},
           {'id': 'learning-rate',
            'term': 'Learning rate',
            'aliases': ['learning rate', 'lr'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A hyperparameter that determines how far parameters move in one update. Too large a value makes training '
                          'unstable; too small a value slows progress.',
            'related': ['epoch', 'overfitting']},
           {'id': 'epoch',
            'term': 'Epoch',
            'aliases': ['epoch'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting.',
            'related': ['overfitting', 'learning-rate']},
           {'id': 'overfitting',
            'term': 'Overfitting',
            'aliases': ['overfitting'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by '
                          'measuring performance on validation data not used for training.',
            'related': ['epoch', 'evaluation-loop']},
           {'id': 'huggingface-hub',
            'term': 'Hugging Face Hub',
            'aliases': ['Hugging Face', 'HF Hub'],
            'lecture': 2,
            'category': 'Fine-tuning',
            'definition': 'A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and '
                          'publish LoRA adapters.',
            'related': ['lora', 'pretraining']},
           {'id': 'ai-agent',
            'term': 'AI agent',
            'aliases': ['Agent'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A system that achieves a goal through a loop in which an LLM decides to call tools, examines the results, and '
                          'decides its next action. Its basic structure is "model + loop + tools".',
            'related': ['agent-loop', 'tool', 'harness']},
           {'id': 'agent-loop',
            'term': 'Agent loop',
            'aliases': ['agent loop', 'tool-calling loop'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'The process of passing input to the model → executing any tool calls returned by the model → returning results '
                          'to the model, repeated until the model gives a final answer. Multiple LLM calls for one question have a '
                          'substantial effect on time, charges, and rate limits.',
            'related': ['ai-agent', 'tool', 'rate-limit']},
           {'id': 'tool',
            'term': 'Tool (Tool use / Function calling)',
            'aliases': ['tool call', 'Function calling', 'Tool use', 'tool'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A function made available for the model to call. The model looks at its name, description, and argument types, '
                          'chooses arguments when needed, and requests a call; the program executes it. Used for search, calculations, '
                          'file operations, external API calls, and more.',
            'related': ['tool-description', 'agent-loop']},
           {'id': 'tool-description',
            'term': 'Tool description',
            'aliases': ['docstring', 'tool specification', 'tool schema'],
            'lecture': 3,
            'category': 'Agents',
            'definition': "Text that explains a tool's purpose and argument meanings to the model. In Pydantic AI, it is generated "
                          'automatically from the function docstring and type hints. Since the model reads it to decide whether and how to '
                          'call a tool, its wording can greatly affect behavior.',
            'related': ['tool', 'pydantic-ai']},
           {'id': 'structured-output',
            'term': 'Structured output',
            'aliases': ['structured output', 'JSON output'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A mechanism for receiving model outputs according to specified types or a JSON schema. This makes programmatic '
                          'postprocessing easier. In Pydantic AI, specify a Pydantic model in output_type.',
            'related': ['pydantic-ai']},
           {'id': 'harness',
            'term': 'Harness',
            'aliases': ['agent harness', 'agent wrapper'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. '
                          'Claude Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, '
                          'edit, and bash.',
            'related': ['coding-agent', 'pi']},
           {'id': 'pi',
            'term': 'Pi',
            'aliases': ['Pi coding agent', 'pi agent'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal '
                          'system prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is '
                          '@earendil-works/pi-coding-agent.',
            'related': ['harness', 'coding-agent']},
           {'id': 'pydantic-ai',
            'term': 'Pydantic AI',
            'aliases': ['pydantic-ai', 'PydanticAI'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A framework for building agents in Python. It uses Pydantic types to define tool arguments and outputs and '
                          'handles many models, such as Gemini and OpenAI, with the same code style. Test models let you check behavior '
                          'without API calls. It is the main framework for Lecture 03.',
            'related': ['test-model', 'structured-output', 'adk']},
           {'id': 'adk',
            'term': 'ADK (Agent Development Kit)',
            'aliases': ['ADK', 'Agent Development Kit', 'google-adk'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web '
                          'lets you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course '
                          'covers it in the Advanced homework.',
            'related': ['pydantic-ai', 'ai-agent']},
           {'id': 'mcp',
            'term': 'MCP (Model Context Protocol)',
            'aliases': ['MCP', 'Model Context Protocol'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through '
                          'an MCP server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you '
                          'will experience this by turning glossary search into an MCP server.',
            'related': ['tool', 'ai-agent', 'mcp-server', 'mcp-client']},
           {'id': 'guardrail',
            'term': 'Guardrail',
            'aliases': ['guardrail'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in '
                          'instructions, restricting tool permissions, checking outputs, and requesting human confirmation before '
                          'important operations.',
            'related': ['human-in-the-loop', 'system-prompt']},
           {'id': 'human-in-the-loop',
            'term': 'Human-in-the-loop',
            'aliases': ['HITL', 'human approval', 'confirmation step'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'Incorporating human confirmation or approval into agent processing. Used before operations that are difficult '
                          'to undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this.',
            'related': ['guardrail', 'tool']},
           {'id': 'test-model',
            'term': 'TestModel',
            'aliases': ['test model'],
            'lecture': 3,
            'category': 'Agents',
            'definition': 'A fake model included in Pydantic AI for testing. It calls tools and returns dummy answers without calling a '
                          'real LLM. It lets you test agent wiring (whether tools are registered, called, and return results) without an '
                          'API key or usage allowance.',
            'related': ['pydantic-ai', 'rate-limit']},
           {'id': 'homework-levels',
            'term': '3 homework levels',
            'aliases': ['3 levels', 'homework levels'],
            'lecture': 1,
            'category': 'Course',
            'definition': 'The structure of AIE2026 homework. There are three levels: "Basic (review lecture content)", "Implementation '
                          '(write your own code)", and "Advanced (research and experimentation)". Completing any one level completes the '
                          'homework for that lecture. As a rule, it is not graded.',
            'related': ['answer-md']},
           {'id': 'answer-md',
            'term': 'answer.md',
            'aliases': ['answer file'],
            'lecture': 1,
            'category': 'Course',
            'definition': 'The file for recording exercise and homework results for each AIE2026 lecture. Located at '
                          'lectures/NN/answer.md. Record the level attempted, what you did, results, and observations, then submit by '
                          'opening a PR from your working branch to main in your own fork.',
            'related': ['pull-request', 'homework-levels']},
           {'id': 'lec-start-tag',
            'term': 'lecNN-start tag',
            'aliases': ['start tag', 'lec03-start'],
            'lecture': 1,
            'category': 'Course',
            'definition': "A Git tag on the starting code for each lecture's exercises in the AIE2026 course repository (for example, "
                          'lec03-start). Even if you have not finished the previous lecture, creating a branch from this tag lets you '
                          "start the current lecture's exercises.",
            'related': ['branch']},
           {'id': 'default-model',
            'term': 'Default model',
            'aliases': ['default AI model', 'gemini-3.5-flash-lite'],
            'lecture': 1,
            'category': 'Course',
            'definition': 'The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate '
                          'limits and stable course operation in mind. You may switch models with the AIE_MODEL environment variable or '
                          'use your own API subscription.',
            'related': ['rate-limit', 'api-key']},
           {'id': 'external-tool',
            'term': 'External tool',
            'aliases': ['external tool', 'external service'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'Using capabilities outside the agent program, such as web retrieval, file operations, databases, and '
                          'third-party APIs, as tools. Lecture 03 function tools run within the same process, whereas external tools are '
                          'called across processes or machines. MCP is a common protocol for this.',
            'related': ['tool', 'mcp', 'mcp-server']},
           {'id': 'mcp-server',
            'term': 'MCP server',
            'aliases': ['MCP server'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'The program that exposes tools (and resources or prompts) through MCP. It exposes tool names, descriptions, and '
                          'argument schemas, executes tools in response to client requests, and returns results. In this course, glossary '
                          'search becomes an MCP server (app/mcp_server.py). Many ready-made servers are available, including fetch, '
                          'filesystem, and GitHub.',
            'related': ['mcp', 'mcp-client', 'mcp-transport', 'fetch-server']},
           {'id': 'mcp-client',
            'term': 'MCP client',
            'aliases': ['MCP client', 'MCP host'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'The component that connects to MCP servers, passes their tools to the model, and calls tools as directed by the '
                          "model. Agent programs (Pydantic AI's MCPToolset) and editors or applications such as Copilot Chat, Claude "
                          'Desktop, and Cursor can serve as clients. A benefit of MCP is using the same server from different clients.',
            'related': ['mcp', 'mcp-server', 'mcp-config']},
           {'id': 'mcp-transport',
            'term': 'Transport (stdio / Streamable HTTP)',
            'aliases': ['stdio', 'Streamable HTTP', 'transport', 'MCP transport'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'The communication channel between an MCP client and server. With stdio, the client starts the server as a '
                          'subprocess and exchanges messages through standard input/output (simple and used on the same machine). With '
                          'Streamable HTTP, the server listens at a URL, allowing connections from other processes or machines (access '
                          'scope and authentication must be considered).',
            'related': ['mcp-server', 'mcp-client']},
           {'id': 'mcp-config',
            'term': 'mcpServers configuration',
            'aliases': ['mcpServers', 'MCP configuration file', 'mcp_servers.json', 'mcp.json'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'A configuration file listing MCP servers to use. For each server, specify a name and a startup command (stdio) '
                          'or destination URL (HTTP). Claude Desktop, Cursor, and Pydantic AI read the same mcpServers format (VS Code / '
                          'Copilot Chat uses servers). This course uses app/mcp_servers.json. Adding tools with just a few configuration '
                          'lines is central to the MCP experience.',
            'related': ['mcp-client', 'tool-namespace']},
           {'id': 'tool-namespace',
            'term': 'Tool namespace',
            'aliases': ['tool-name prefix', 'prefix', 'name collision'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'Adding a prefix such as "server_name_tool_name" to distinguish identically named tools when using multiple MCP '
                          "servers. This course's app/agent.py automatically adds the server name from the configuration (for example, "
                          "glossary_search_glossary; Pydantic AI's load_mcp_toolsets does the same). More tools mean more descriptions "
                          'sent to the model, affecting tool-selection accuracy and token counts.',
            'related': ['mcp-config', 'tool-description', 'rate-limit']},
           {'id': 'fetch-server',
            'term': 'fetch server',
            'aliases': ['mcp-server-fetch', 'fetch', 'web retrieval tool'],
            'lecture': 5,
            'category': 'MCP',
            'definition': 'A ready-made MCP server that retrieves a specified URL and returns the page body as Markdown (written in '
                          'Python; started with uvx mcp-server-fetch). This course uses it as an example of "adding an external tool just '
                          'through configuration". It actually accesses URLs chosen by the model, so users must understand where it sends '
                          'data and what it sends.',
            'related': ['mcp-server', 'external-tool', 'guardrail']}]}

SFT_SHA256 = 'df11fef8e6251754ff7c814ed2f7d979b96bd927a252aef36819d1ddf361386f'  # sha256 of glossary_sft.jsonl produced by scripts/make_sft_dataset.py
SFT_JSONL_LINES = [
    '{"messages": [{"role": "user", "content": "What is AI engineering?"}, {"role": "assistant", "content": "The technical field of designing, building, evaluating, and operating useful applications and systems using pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, connecting them to data and tools, and iterating on evaluation and improvement, rather than building models from scratch."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain AI engineering."}, {"role": "assistant", "content": "The technical field of designing, building, evaluating, and operating useful applications and systems using pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, connecting them to data and tools, and iterating on evaluation and improvement, rather than building models from scratch."}]}',
    '{"messages": [{"role": "user", "content": "What is Evaluation and improvement cycle?"}, {"role": "assistant", "content": "A development approach that repeats build → measure → improve. Since AI application outputs vary probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a learning support agent."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Evaluation and improvement cycle."}, {"role": "assistant", "content": "A development approach that repeats build → measure → improve. Since AI application outputs vary probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a learning support agent."}]}',
    '{"messages": [{"role": "user", "content": "What is LLM (Large Language Model)?"}, {"role": "assistant", "content": "A model trained on large amounts of text that generates text by predicting the tokens following its input. It can handle many tasks, including question answering, summarization, and code generation, simply by changing the instructions (prompt)."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain LLM (Large Language Model)."}, {"role": "assistant", "content": "A model trained on large amounts of text that generates text by predicting the tokens following its input. It can handle many tasks, including question answering, summarization, and code generation, simply by changing the instructions (prompt)."}]}',
    '{"messages": [{"role": "user", "content": "What is Token?"}, {"role": "assistant", "content": "The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; for example, Japanese tends to require more tokens per character than English."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Token."}, {"role": "assistant", "content": "The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; for example, Japanese tends to require more tokens per character than English."}]}',
    '{"messages": [{"role": "user", "content": "What is Context window?"}, {"role": "assistant", "content": "The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and tool results must also fit within this limit, making the choice of what to pass to the model important."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Context window."}, {"role": "assistant", "content": "The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and tool results must also fit within this limit, making the choice of what to pass to the model important."}]}',
    '{"messages": [{"role": "user", "content": "What is Prompt?"}, {"role": "assistant", "content": "Input text passed to an LLM. In addition to questions, it can include roles, constraints, output formats, and examples to guide the output."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Prompt."}, {"role": "assistant", "content": "Input text passed to an LLM. In addition to questions, it can include roles, constraints, output formats, and examples to guide the output."}]}',
    '{"messages": [{"role": "user", "content": "What is System prompt?"}, {"role": "assistant", "content": "Input that specifies model behavior (role, tone, rules, and so on) separately from the user\'s question. In Pydantic AI, it is provided through the Agent instructions argument."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain System prompt."}, {"role": "assistant", "content": "Input that specifies model behavior (role, tone, rules, and so on) separately from the user\'s question. In Pydantic AI, it is provided through the Agent instructions argument."}]}',
    '{"messages": [{"role": "user", "content": "What is temperature?"}, {"role": "assistant", "content": "A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain temperature."}, {"role": "assistant", "content": "A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers."}]}',
    '{"messages": [{"role": "user", "content": "What is Hallucination?"}, {"role": "assistant", "content": "The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Hallucination."}, {"role": "assistant", "content": "The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely when asked about things the model does not know, such as recent information or organization- or course-specific details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors through evaluation."}]}',
    '{"messages": [{"role": "user", "content": "What is API?"}, {"role": "assistant", "content": "An interface through which a program calls the functionality of another service. In this course, we use an LLM by calling the Gemini API from Python."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain API."}, {"role": "assistant", "content": "An interface through which a program calls the functionality of another service. In this course, we use an LLM by calling the Gemini API from Python."}]}',
    '{"messages": [{"role": "user", "content": "What is API key?"}, {"role": "assistant", "content": "A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as a Codespaces secret and read it through the GEMINI_API_KEY environment variable."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain API key."}, {"role": "assistant", "content": "A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as a Codespaces secret and read it through the GEMINI_API_KEY environment variable."}]}',
    '{"messages": [{"role": "user", "content": "What is Environment variable?"}, {"role": "assistant", "content": "A value set outside a program (in the OS or execution environment) that the program can read by name. Used to separate secrets such as API keys and environment-specific settings from code."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Environment variable."}, {"role": "assistant", "content": "A value set outside a program (in the OS or execution environment) that the program can read by name. Used to separate secrets such as API keys and environment-specific settings from code."}]}',
    '{"messages": [{"role": "user", "content": "What is Rate limit?"}, {"role": "assistant", "content": "A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have particularly strict limits, and agents reach them easily because they call the LLM multiple times for one question."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Rate limit."}, {"role": "assistant", "content": "A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have particularly strict limits, and agents reach them easily because they call the LLM multiple times for one question."}]}',
    '{"messages": [{"role": "user", "content": "What is GitHub Codespaces?"}, {"role": "assistant", "content": "A cloud development environment provided by GitHub. Open VS Code in a browser and immediately use an environment configured by the repository. Personal accounts have a monthly free allowance; if you do not stop the Codespace when idle, it continues consuming that allowance."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain GitHub Codespaces."}, {"role": "assistant", "content": "A cloud development environment provided by GitHub. Open VS Code in a browser and immediately use an environment configured by the repository. Personal accounts have a monthly free allowance; if you do not stop the Codespace when idle, it continues consuming that allowance."}]}',
    '{"messages": [{"role": "user", "content": "What is devcontainer?"}, {"role": "assistant", "content": "A mechanism for defining a development environment as a container. Specify the Python version, extensions, and startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain devcontainer."}, {"role": "assistant", "content": "A mechanism for defining a development environment as a container. Specify the Python version, extensions, and startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment."}]}',
    '{"messages": [{"role": "user", "content": "What is Codespaces secret?"}, {"role": "assistant", "content": "A secret value registered in your GitHub account settings. It is passed as an environment variable when you start Codespaces for a selected repository. When registering it, select the repository you will use (your own fork). If registered while a Codespace is running, restart the Codespace."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Codespaces secret."}, {"role": "assistant", "content": "A secret value registered in your GitHub account settings. It is passed as an environment variable when you start Codespaces for a selected repository. When registering it, select the repository you will use (your own fork). If registered while a Codespace is running, restart the Codespace."}]}',
    '{"messages": [{"role": "user", "content": "What is fork?"}, {"role": "assistant", "content": "Copying someone else\'s entire repository to your own account. In this course, fork the course repository and work and submit within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain fork."}, {"role": "assistant", "content": "Copying someone else\'s entire repository to your own account. In this course, fork the course repository and work and submit within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "What is Branch?"}, {"role": "assistant", "content": "A mechanism for separating streams of work. Try changes without breaking main, then merge them into main through a PR once ready. In this course, create a working branch such as lec01 for each lecture."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Branch."}, {"role": "assistant", "content": "A mechanism for separating streams of work. Try changes without breaking main, then merge them into main through a PR once ready. In this course, create a working branch such as lec01 for each lecture."}]}',
    '{"messages": [{"role": "user", "content": "What is Pull Request (PR)?"}, {"role": "assistant", "content": "A mechanism for requesting that changes from one branch be merged into another. It supports reviewing differences, comments, and displaying automated test results. Submit work for this course as a PR from your working branch to main within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pull Request (PR)."}, {"role": "assistant", "content": "A mechanism for requesting that changes from one branch be merged into another. It supports reviewing differences, comments, and displaying automated test results. Submit work for this course as a PR from your working branch to main within your own fork."}]}',
    '{"messages": [{"role": "user", "content": "What is GitHub Copilot?"}, {"role": "assistant", "content": "GitHub\'s AI coding assistance. It supports code completion in the editor, questions in chat, and editing multiple files in Agent mode. Copilot Free limits completion counts and chat usage."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain GitHub Copilot."}, {"role": "assistant", "content": "GitHub\'s AI coding assistance. It supports code completion in the editor, questions in chat, and editing multiple files in Agent mode. Copilot Free limits completion counts and chat usage."}]}',
    '{"messages": [{"role": "user", "content": "What is Code completion?"}, {"role": "assistant", "content": "A feature in which AI suggests code to follow partially written code or comments. Accept suggestions with Tab or a similar action. Suggestions may be incorrect, so read and understand them before accepting."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Code completion."}, {"role": "assistant", "content": "A feature in which AI suggests code to follow partially written code or comments. Accept suggestions with Tab or a similar action. Suggestions may be incorrect, so read and understand them before accepting."}]}',
    '{"messages": [{"role": "user", "content": "What is Coding agent?"}, {"role": "assistant", "content": "AI that autonomously carries out requested programming work over multiple steps, using tools to read and write files and run commands. Examples include Claude Code, Copilot Agent mode, and Pi. Internally, it has the same LLM + loop + tools structure as an AI agent."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Coding agent."}, {"role": "assistant", "content": "AI that autonomously carries out requested programming work over multiple steps, using tools to read and write files and run commands. Examples include Claude Code, Copilot Agent mode, and Pi. Internally, it has the same LLM + loop + tools structure as an AI agent."}]}',
    '{"messages": [{"role": "user", "content": "What is Pretraining?"}, {"role": "assistant", "content": "The initial stage of training a model to predict the next token using large amounts of general-purpose text. Much of its language knowledge and general common sense is acquired here."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pretraining."}, {"role": "assistant", "content": "The initial stage of training a model to predict the next token using large amounts of general-purpose text. Much of its language knowledge and general common sense is acquired here."}]}',
    '{"messages": [{"role": "user", "content": "What is Fine-tuning?"}, {"role": "assistant", "content": "Further training a pretrained model on data suited to a specific task or format. It is suitable for teaching output format, tone, or behavior in a specific domain. For up-to-date facts, consulting external data through tools or RAG is often more suitable."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Fine-tuning."}, {"role": "assistant", "content": "Further training a pretrained model on data suited to a specific task or format. It is suitable for teaching output format, tone, or behavior in a specific domain. For up-to-date facts, consulting external data through tools or RAG is often more suitable."}]}',
    '{"messages": [{"role": "user", "content": "What is Supervised Fine-Tuning (SFT)?"}, {"role": "assistant", "content": "A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. A representative method for teaching instruction-following behavior."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Supervised Fine-Tuning (SFT)."}, {"role": "assistant", "content": "A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. A representative method for teaching instruction-following behavior."}]}',
    '{"messages": [{"role": "user", "content": "What is PEFT?"}, {"role": "assistant", "content": "An umbrella term for methods that train only a small subset of model parameters or a small set of added parameters instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a representative example."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain PEFT."}, {"role": "assistant", "content": "An umbrella term for methods that train only a small subset of model parameters or a small set of added parameters instead of all parameters. This can greatly reduce GPU memory and storage requirements. LoRA is a representative example."}]}',
    '{"messages": [{"role": "user", "content": "What is LoRA?"}, {"role": "assistant", "content": "A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain LoRA."}, {"role": "assistant", "content": "A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."}]}',
    '{"messages": [{"role": "user", "content": "What is Rank (LoRA r)?"}, {"role": "assistant", "content": "A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases expressive capacity, but also increases the number of trainable parameters and memory use."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Rank (LoRA r)."}, {"role": "assistant", "content": "A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases expressive capacity, but also increases the number of trainable parameters and memory use."}]}',
    '{"messages": [{"role": "user", "content": "What is QLoRA?"}, {"role": "assistant", "content": "A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA and compare memory and output differences when loading in 4-bit in the advanced section."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain QLoRA."}, {"role": "assistant", "content": "A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA and compare memory and output differences when loading in 4-bit in the advanced section."}]}',
    '{"messages": [{"role": "user", "content": "What is Quantization?"}, {"role": "assistant", "content": "A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce memory use and computation. Accuracy may decrease slightly."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Quantization."}, {"role": "assistant", "content": "A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce memory use and computation. Accuracy may decrease slightly."}]}',
    '{"messages": [{"role": "user", "content": "What is Learning rate?"}, {"role": "assistant", "content": "A hyperparameter that determines how far parameters move in one update. Too large a value makes training unstable; too small a value slows progress."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Learning rate."}, {"role": "assistant", "content": "A hyperparameter that determines how far parameters move in one update. Too large a value makes training unstable; too small a value slows progress."}]}',
    '{"messages": [{"role": "user", "content": "What is Epoch?"}, {"role": "assistant", "content": "One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Epoch."}, {"role": "assistant", "content": "One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting."}]}',
    '{"messages": [{"role": "user", "content": "What is Overfitting?"}, {"role": "assistant", "content": "A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Overfitting."}, {"role": "assistant", "content": "A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training."}]}',
    '{"messages": [{"role": "user", "content": "What is Hugging Face Hub?"}, {"role": "assistant", "content": "A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and publish LoRA adapters."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Hugging Face Hub."}, {"role": "assistant", "content": "A platform for sharing and downloading public models and datasets. Used to obtain open-weight models and publish LoRA adapters."}]}',
    '{"messages": [{"role": "user", "content": "What is AI agent?"}, {"role": "assistant", "content": "A system that achieves a goal through a loop in which an LLM decides to call tools, examines the results, and decides its next action. Its basic structure is \\"model + loop + tools\\"."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain AI agent."}, {"role": "assistant", "content": "A system that achieves a goal through a loop in which an LLM decides to call tools, examines the results, and decides its next action. Its basic structure is \\"model + loop + tools\\"."}]}',
    '{"messages": [{"role": "user", "content": "What is Agent loop?"}, {"role": "assistant", "content": "The process of passing input to the model → executing any tool calls returned by the model → returning results to the model, repeated until the model gives a final answer. Multiple LLM calls for one question have a substantial effect on time, charges, and rate limits."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Agent loop."}, {"role": "assistant", "content": "The process of passing input to the model → executing any tool calls returned by the model → returning results to the model, repeated until the model gives a final answer. Multiple LLM calls for one question have a substantial effect on time, charges, and rate limits."}]}',
    '{"messages": [{"role": "user", "content": "What is Tool (Tool use / Function calling)?"}, {"role": "assistant", "content": "A function made available for the model to call. The model looks at its name, description, and argument types, chooses arguments when needed, and requests a call; the program executes it. Used for search, calculations, file operations, external API calls, and more."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Tool (Tool use / Function calling)."}, {"role": "assistant", "content": "A function made available for the model to call. The model looks at its name, description, and argument types, chooses arguments when needed, and requests a call; the program executes it. Used for search, calculations, file operations, external API calls, and more."}]}',
    '{"messages": [{"role": "user", "content": "What is Tool description?"}, {"role": "assistant", "content": "Text that explains a tool\'s purpose and argument meanings to the model. In Pydantic AI, it is generated automatically from the function docstring and type hints. Since the model reads it to decide whether and how to call a tool, its wording can greatly affect behavior."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Tool description."}, {"role": "assistant", "content": "Text that explains a tool\'s purpose and argument meanings to the model. In Pydantic AI, it is generated automatically from the function docstring and type hints. Since the model reads it to decide whether and how to call a tool, its wording can greatly affect behavior."}]}',
    '{"messages": [{"role": "user", "content": "What is Structured output?"}, {"role": "assistant", "content": "A mechanism for receiving model outputs according to specified types or a JSON schema. This makes programmatic postprocessing easier. In Pydantic AI, specify a Pydantic model in output_type."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Structured output."}, {"role": "assistant", "content": "A mechanism for receiving model outputs according to specified types or a JSON schema. This makes programmatic postprocessing easier. In Pydantic AI, specify a Pydantic model in output_type."}]}',
    '{"messages": [{"role": "user", "content": "What is Harness?"}, {"role": "assistant", "content": "A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. Claude Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, edit, and bash."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Harness."}, {"role": "assistant", "content": "A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. Claude Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, edit, and bash."}]}',
    '{"messages": [{"role": "user", "content": "What is Pi?"}, {"role": "assistant", "content": "An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal system prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is @earendil-works/pi-coding-agent."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pi."}, {"role": "assistant", "content": "An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal system prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is @earendil-works/pi-coding-agent."}]}',
    '{"messages": [{"role": "user", "content": "What is Pydantic AI?"}, {"role": "assistant", "content": "A framework for building agents in Python. It uses Pydantic types to define tool arguments and outputs and handles many models, such as Gemini and OpenAI, with the same code style. Test models let you check behavior without API calls. It is the main framework for Lecture 03."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Pydantic AI."}, {"role": "assistant", "content": "A framework for building agents in Python. It uses Pydantic types to define tool arguments and outputs and handles many models, such as Gemini and OpenAI, with the same code style. Test models let you check behavior without API calls. It is the main framework for Lecture 03."}]}',
    '{"messages": [{"role": "user", "content": "What is ADK (Agent Development Kit)?"}, {"role": "assistant", "content": "An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web lets you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course covers it in the Advanced homework."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain ADK (Agent Development Kit)."}, {"role": "assistant", "content": "An open-source agent development framework released by Google. It integrates closely with Gemini, and adk web lets you visually follow tool calls. Version 2.0 introduces breaking changes from the 1.x series. This course covers it in the Advanced homework."}]}',
    '{"messages": [{"role": "user", "content": "What is MCP (Model Context Protocol)?"}, {"role": "assistant", "content": "An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will experience this by turning glossary search into an MCP server."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain MCP (Model Context Protocol)."}, {"role": "assistant", "content": "An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will experience this by turning glossary search into an MCP server."}]}',
    '{"messages": [{"role": "user", "content": "What is Guardrail?"}, {"role": "assistant", "content": "A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in instructions, restricting tool permissions, checking outputs, and requesting human confirmation before important operations."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Guardrail."}, {"role": "assistant", "content": "A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in instructions, restricting tool permissions, checking outputs, and requesting human confirmation before important operations."}]}',
    '{"messages": [{"role": "user", "content": "What is Human-in-the-loop?"}, {"role": "assistant", "content": "Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Human-in-the-loop."}, {"role": "assistant", "content": "Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this."}]}',
    '{"messages": [{"role": "user", "content": "What is TestModel?"}, {"role": "assistant", "content": "A fake model included in Pydantic AI for testing. It calls tools and returns dummy answers without calling a real LLM. It lets you test agent wiring (whether tools are registered, called, and return results) without an API key or usage allowance."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain TestModel."}, {"role": "assistant", "content": "A fake model included in Pydantic AI for testing. It calls tools and returns dummy answers without calling a real LLM. It lets you test agent wiring (whether tools are registered, called, and return results) without an API key or usage allowance."}]}',
    '{"messages": [{"role": "user", "content": "What is 3 homework levels?"}, {"role": "assistant", "content": "The structure of AIE2026 homework. There are three levels: \\"Basic (review lecture content)\\", \\"Implementation (write your own code)\\", and \\"Advanced (research and experimentation)\\". Completing any one level completes the homework for that lecture. As a rule, it is not graded."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain 3 homework levels."}, {"role": "assistant", "content": "The structure of AIE2026 homework. There are three levels: \\"Basic (review lecture content)\\", \\"Implementation (write your own code)\\", and \\"Advanced (research and experimentation)\\". Completing any one level completes the homework for that lecture. As a rule, it is not graded."}]}',
    '{"messages": [{"role": "user", "content": "What is answer.md?"}, {"role": "assistant", "content": "The file for recording exercise and homework results for each AIE2026 lecture. Located at lectures/NN/answer.md. Record the level attempted, what you did, results, and observations, then submit by opening a PR from your working branch to main in your own fork."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain answer.md."}, {"role": "assistant", "content": "The file for recording exercise and homework results for each AIE2026 lecture. Located at lectures/NN/answer.md. Record the level attempted, what you did, results, and observations, then submit by opening a PR from your working branch to main in your own fork."}]}',
    '{"messages": [{"role": "user", "content": "What is lecNN-start tag?"}, {"role": "assistant", "content": "A Git tag on the starting code for each lecture\'s exercises in the AIE2026 course repository (for example, lec03-start). Even if you have not finished the previous lecture, creating a branch from this tag lets you start the current lecture\'s exercises."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain lecNN-start tag."}, {"role": "assistant", "content": "A Git tag on the starting code for each lecture\'s exercises in the AIE2026 course repository (for example, lec03-start). Even if you have not finished the previous lecture, creating a branch from this tag lets you start the current lecture\'s exercises."}]}',
    '{"messages": [{"role": "user", "content": "What is Default model?"}, {"role": "assistant", "content": "The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API subscription."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Default model."}, {"role": "assistant", "content": "The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API subscription."}]}',
    '{"messages": [{"role": "user", "content": "What is External tool?"}, {"role": "assistant", "content": "Using capabilities outside the agent program, such as web retrieval, file operations, databases, and third-party APIs, as tools. Lecture 03 function tools run within the same process, whereas external tools are called across processes or machines. MCP is a common protocol for this."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain External tool."}, {"role": "assistant", "content": "Using capabilities outside the agent program, such as web retrieval, file operations, databases, and third-party APIs, as tools. Lecture 03 function tools run within the same process, whereas external tools are called across processes or machines. MCP is a common protocol for this."}]}',
    '{"messages": [{"role": "user", "content": "What is MCP server?"}, {"role": "assistant", "content": "The program that exposes tools (and resources or prompts) through MCP. It exposes tool names, descriptions, and argument schemas, executes tools in response to client requests, and returns results. In this course, glossary search becomes an MCP server (app/mcp_server.py). Many ready-made servers are available, including fetch, filesystem, and GitHub."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain MCP server."}, {"role": "assistant", "content": "The program that exposes tools (and resources or prompts) through MCP. It exposes tool names, descriptions, and argument schemas, executes tools in response to client requests, and returns results. In this course, glossary search becomes an MCP server (app/mcp_server.py). Many ready-made servers are available, including fetch, filesystem, and GitHub."}]}',
    '{"messages": [{"role": "user", "content": "What is MCP client?"}, {"role": "assistant", "content": "The component that connects to MCP servers, passes their tools to the model, and calls tools as directed by the model. Agent programs (Pydantic AI\'s MCPToolset) and editors or applications such as Copilot Chat, Claude Desktop, and Cursor can serve as clients. A benefit of MCP is using the same server from different clients."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain MCP client."}, {"role": "assistant", "content": "The component that connects to MCP servers, passes their tools to the model, and calls tools as directed by the model. Agent programs (Pydantic AI\'s MCPToolset) and editors or applications such as Copilot Chat, Claude Desktop, and Cursor can serve as clients. A benefit of MCP is using the same server from different clients."}]}',
    '{"messages": [{"role": "user", "content": "What is Transport (stdio / Streamable HTTP)?"}, {"role": "assistant", "content": "The communication channel between an MCP client and server. With stdio, the client starts the server as a subprocess and exchanges messages through standard input/output (simple and used on the same machine). With Streamable HTTP, the server listens at a URL, allowing connections from other processes or machines (access scope and authentication must be considered)."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Transport (stdio / Streamable HTTP)."}, {"role": "assistant", "content": "The communication channel between an MCP client and server. With stdio, the client starts the server as a subprocess and exchanges messages through standard input/output (simple and used on the same machine). With Streamable HTTP, the server listens at a URL, allowing connections from other processes or machines (access scope and authentication must be considered)."}]}',
    '{"messages": [{"role": "user", "content": "What is mcpServers configuration?"}, {"role": "assistant", "content": "A configuration file listing MCP servers to use. For each server, specify a name and a startup command (stdio) or destination URL (HTTP). Claude Desktop, Cursor, and Pydantic AI read the same mcpServers format (VS Code / Copilot Chat uses servers). This course uses app/mcp_servers.json. Adding tools with just a few configuration lines is central to the MCP experience."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain mcpServers configuration."}, {"role": "assistant", "content": "A configuration file listing MCP servers to use. For each server, specify a name and a startup command (stdio) or destination URL (HTTP). Claude Desktop, Cursor, and Pydantic AI read the same mcpServers format (VS Code / Copilot Chat uses servers). This course uses app/mcp_servers.json. Adding tools with just a few configuration lines is central to the MCP experience."}]}',
    '{"messages": [{"role": "user", "content": "What is Tool namespace?"}, {"role": "assistant", "content": "Adding a prefix such as \\"server_name_tool_name\\" to distinguish identically named tools when using multiple MCP servers. This course\'s app/agent.py automatically adds the server name from the configuration (for example, glossary_search_glossary; Pydantic AI\'s load_mcp_toolsets does the same). More tools mean more descriptions sent to the model, affecting tool-selection accuracy and token counts."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain Tool namespace."}, {"role": "assistant", "content": "Adding a prefix such as \\"server_name_tool_name\\" to distinguish identically named tools when using multiple MCP servers. This course\'s app/agent.py automatically adds the server name from the configuration (for example, glossary_search_glossary; Pydantic AI\'s load_mcp_toolsets does the same). More tools mean more descriptions sent to the model, affecting tool-selection accuracy and token counts."}]}',
    '{"messages": [{"role": "user", "content": "What is fetch server?"}, {"role": "assistant", "content": "A ready-made MCP server that retrieves a specified URL and returns the page body as Markdown (written in Python; started with uvx mcp-server-fetch). This course uses it as an example of \\"adding an external tool just through configuration\\". It actually accesses URLs chosen by the model, so users must understand where it sends data and what it sends."}]}',
    '{"messages": [{"role": "user", "content": "Briefly explain fetch server."}, {"role": "assistant", "content": "A ready-made MCP server that retrieves a specified URL and returns the page body as Markdown (written in Python; started with uvx mcp-server-fetch). This course uses it as an example of \\"adding an external tool just through configuration\\". It actually accesses URLs chosen by the model, so users must understand where it sends data and what it sends."}]}',
]

EVAL_SET = {'abstain_answer': 'That term is not in the course glossary.',
 'train_absent_terms': ['Direct Preference Optimization',
                        'speculative decoding',
                        'KV cache',
                        'perplexity',
                        'diffusion model',
                        'batch normalization',
                        'gradient checkpointing',
                        'BLEU score'],
 'items': [{'group': 'in_glossary',
            'question': 'What is LLM (Large Language Model)?',
            'term_id': 'llm',
            'term': 'LLM (Large Language Model)',
            'gold': 'A model trained on large amounts of text that generates text by predicting the tokens following its input. It can '
                    'handle many tasks, including question answering, summarization, and code generation, simply by changing the '
                    'instructions (prompt).',
            'must_include': ['predicting', 'summarization'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain Context window.',
            'term_id': 'context-window',
            'term': 'Context window',
            'gold': 'The maximum total number of input and output tokens an LLM can handle in one call. Conversation history and tool '
                    'results must also fit within this limit, making the choice of what to pass to the model important.',
            'must_include': ['one call', 'tool results'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Hallucination?',
            'term_id': 'hallucination',
            'term': 'Hallucination',
            'gold': 'The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely '
                    'when asked about things the model does not know, such as recent information or organization- or course-specific '
                    'details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors '
                    'through evaluation.',
            'must_include': ['course-specific', 'external data'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is LoRA?',
            'term_id': 'lora',
            'term': 'LoRA',
            'gold': 'A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices '
                    '(low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) '
                    'in small files. The Lecture 02 exercises cover LoRA.',
            'must_include': ['low-rank', 'adapters', 'Lecture 02'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain QLoRA.',
            'term_id': 'qlora',
            'term': 'QLoRA',
            'gold': 'A method that saves memory by quantizing base-model weights to, for example, 4 bits, then trains with LoRA. This lets '
                    'smaller GPUs handle larger models. In the Lecture 02 exercises, the model fits in GPU memory, so we train with LoRA '
                    'and compare memory and output differences when loading in 4-bit in the advanced section.',
            'must_include': ['4 bits', 'smaller GPUs'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Overfitting?',
            'term_id': 'overfitting',
            'term': 'Overfitting',
            'gold': 'A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring '
                    'performance on validation data not used for training.',
            'must_include': ['validation data', 'unseen'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is MCP (Model Context Protocol)?',
            'term_id': 'mcp',
            'term': 'MCP (Model Context Protocol)',
            'gold': 'An open standard protocol connecting AI applications to external tools and data sources. Tools exposed through an MCP '
                    'server can be used in a common way from compatible agents and editors (MCP clients). In Lecture 05, you will '
                    'experience this by turning glossary search into an MCP server.',
            'must_include': ['open standard', 'Lecture 05'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain Guardrail.',
            'term_id': 'guardrail',
            'term': 'Guardrail',
            'gold': 'A constraint or check designed to prevent undesirable agent behavior or outputs. Methods include rules in '
                    'instructions, restricting tool permissions, checking outputs, and requesting human confirmation before important '
                    'operations.',
            'must_include': ['tool permissions', 'human confirmation'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Harness?',
            'term_id': 'harness',
            'term': 'Harness',
            'gold': 'A program that surrounds an LLM with a loop, tools, a user interface (UI), and so on to enable actual work. Claude '
                    'Code and Pi are examples. Pi is known as a thin harness with just four built-in tools: read, write, edit, and bash.',
            'must_include': ['Claude Code', 'four built-in tools'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Pi?',
            'term_id': 'pi',
            'term': 'Pi',
            'gold': 'An open-source (MIT-licensed) coding agent that runs in the terminal. It has minimal features and a minimal system '
                    'prompt, designed for users to extend. Version 1.0 was released in October 2026. Its npm package name is '
                    '@earendil-works/pi-coding-agent.',
            'must_include': ['MIT', 'terminal', '@earendil-works/pi-coding-agent'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'What is Default model?',
            'term_id': 'default-model',
            'term': 'Default model',
            'gold': 'The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and '
                    'stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API '
                    'subscription.',
            'must_include': ['gemini-3.5-flash-lite', 'AIE_MODEL'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'in_glossary',
            'question': 'Briefly explain answer.md.',
            'term_id': 'answer-md',
            'term': 'answer.md',
            'gold': 'The file for recording exercise and homework results for each AIE2026 lecture. Located at lectures/NN/answer.md. '
                    'Record the level attempted, what you did, results, and observations, then submit by opening a PR from your working '
                    'branch to main in your own fork.',
            'must_include': ['lectures/NN/answer.md', 'working branch'],
            'why': 'Exact training template; measures memorization of a seen question.'},
           {'group': 'paraphrased',
            'question': 'What does HITL mean in agent design?',
            'term_id': 'human-in-the-loop',
            'term': 'Human-in-the-loop',
            'gold': 'Incorporating human confirmation or approval into agent processing. Used before operations that are difficult to '
                    'undo, such as writing or sending data. In Pydantic AI, tools with requires_approval=True support this.',
            'must_include': ['difficult to undo', 'requires_approval'],
            'why': "Alias (acronym) plus extra context words; also contains the generic alias 'agent', which tempts a naive search."},
           {'group': 'paraphrased',
            'question': 'In LoRA, what does the r hyperparameter control?',
            'term_id': 'lora-rank',
            'term': 'Rank (LoRA r)',
            'gold': 'A hyperparameter that determines the size of the low-rank matrices used in LoRA. A larger value increases expressive '
                    'capacity, but also increases the number of trainable parameters and memory use.',
            'must_include': ['expressive capacity', 'trainable parameters'],
            'why': "Asks about the one-letter alias 'r' inside a LoRA question; tests whether the right entry (rank, not LoRA) is chosen."},
           {'group': 'paraphrased',
            'question': 'Why would I get a 429 error from an LLM API?',
            'term_id': 'rate-limit',
            'term': 'Rate limit',
            'gold': 'A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers '
                    'have particularly strict limits, and agents reach them easily because they call the LLM multiple times for one '
                    'question.',
            'must_include': ['free tiers', 'multiple times'],
            'why': "Symptom-style question: the alias '429 error' appears, but the term 'rate limit' never does."},
           {'group': 'paraphrased',
            'question': 'What does Low-Rank Adaptation do to a pretrained model?',
            'term_id': 'lora',
            'term': 'LoRA',
            'gold': 'A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices '
                    '(low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) '
                    'in small files. The Lecture 02 exercises cover LoRA.',
            'must_include': ['adapters', 'Lecture 02'],
            'why': 'Long-form alias with a new question frame; course-specific keywords check it is the glossary answer, not general '
                   'knowledge.'},
           {'group': 'paraphrased',
            'question': 'How do I pass a Codespaces secret to my code, and what if my Codespace is already running?',
            'term_id': 'codespaces-secret',
            'term': 'Codespaces secret',
            'gold': 'A secret value registered in your GitHub account settings. It is passed as an environment variable when you start '
                    'Codespaces for a selected repository. When registering it, select the repository you will use (your own fork). If '
                    'registered while a Codespace is running, restart the Codespace.',
            'must_include': ['environment variable', 'restart', 'own fork'],
            'why': "Two-part practical question that overlaps with the more generic 'Codespaces' entry."},
           {'group': 'paraphrased',
            'question': 'What goes into devcontainer.json?',
            'term_id': 'devcontainer',
            'term': 'devcontainer',
            'gold': 'A mechanism for defining a development environment as a container. Specify the Python version, extensions, and '
                    'startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets the same environment.',
            'must_include': ['Python version', 'startup commands'],
            'why': "Asks via the file-name alias and about contents rather than 'what is'."},
           {'group': 'paraphrased',
            'question': 'What is instruction tuning?',
            'term_id': 'sft',
            'term': 'Supervised Fine-Tuning (SFT)',
            'gold': 'A method of further training a model using pairs of inputs and desired outputs, such as questions and model answers. '
                    'A representative method for teaching instruction-following behavior.',
            'must_include': ['pairs', 'instruction-following'],
            'why': 'Synonym alias that never names SFT; general knowledge may give a different (broader) definition.'},
           {'group': 'paraphrased',
            'question': 'Walk me through the tool-calling loop of an agent.',
            'term_id': 'agent-loop',
            'term': 'Agent loop',
            'gold': 'The process of passing input to the model → executing any tool calls returned by the model → returning results to the '
                    'model, repeated until the model gives a final answer. Multiple LLM calls for one question have a substantial effect '
                    'on time, charges, and rate limits.',
            'must_include': ['final answer', 'rate limits'],
            'why': "Imperative wording with the alias 'tool-calling loop'; competes with 'tool' and 'agent' entries."},
           {'group': 'paraphrased',
            'question': 'Which model do the AIE2026 exercises use by default?',
            'term_id': 'default-model',
            'term': 'Default model',
            'gold': 'The standard model for AIE2026 exercises and homework is gemini-3.5-flash-lite, chosen with free-tier rate limits and '
                    'stable course operation in mind. You may switch models with the AIE_MODEL environment variable or use your own API '
                    'subscription.',
            'must_include': ['gemini-3.5-flash-lite', 'AIE_MODEL'],
            'why': 'Purely course-specific fact asked without the term or any alias; only memorized knowledge can answer it.'},
           {'group': 'paraphrased',
            'question': 'What should I avoid doing with my API key, and how is it read in this course?',
            'term_id': 'api-key',
            'term': 'API key',
            'gold': 'A secret string that identifies the person calling an API. If it leaks, someone else can consume your usage '
                    'allowance, so do not put it directly in code, commit it, or paste it into chat. In this course, register it as a '
                    'Codespaces secret and read it through the GEMINI_API_KEY environment variable.',
            'must_include': ['commit', 'GEMINI_API_KEY'],
            'why': 'Advice-style question; GEMINI_API_KEY is a course-specific detail a generic answer would miss.'},
           {'group': 'paraphrased',
            'question': 'Explain quantisation of model weights.',
            'term_id': 'quantization',
            'term': 'Quantization',
            'gold': 'A technique for representing model weights with fewer bits (for example, from 16 bits to 4 bits) to reduce memory use '
                    'and computation. Accuracy may decrease slightly.',
            'must_include': ['fewer bits', '4 bits'],
            'why': "British spelling ('quantisation'): no exact match, tests the fuzzy fallback and spelling robustness."},
           {'group': 'paraphrased',
            'question': 'Why do LLMs sometimes confidently make up facts?',
            'term_id': 'hallucination',
            'term': 'Hallucination',
            'gold': 'The phenomenon of an LLM generating plausible content that is factually incorrect or nonexistent. It is more likely '
                    'when asked about things the model does not know, such as recent information or organization- or course-specific '
                    'details. Mitigations include consulting external data through tools, asking for evidence, and detecting errors '
                    'through evaluation.',
            'must_include': ['course-specific', 'external data'],
            'why': 'Describes the phenomenon without naming it; keyword search has nothing to match.'},
           {'group': 'paraphrased',
            'question': 'How do I avoid name collisions between tools from multiple MCP servers?',
            'term_id': 'tool-namespace',
            'term': 'Tool namespace',
            'gold': 'Adding a prefix such as "server_name_tool_name" to distinguish identically named tools when using multiple MCP '
                    "servers. This course's app/agent.py automatically adds the server name from the configuration (for example, "
                    "glossary_search_glossary; Pydantic AI's load_mcp_toolsets does the same). More tools mean more descriptions sent to "
                    'the model, affecting tool-selection accuracy and token counts.',
            'must_include': ['prefix', 'server_name_tool_name'],
            'why': "Plural of the alias 'name collision' plus the competing term 'MCP'; tests problem-to-term mapping."},
           {'group': 'absent',
            'question': 'What is RAG (retrieval-augmented generation)?',
            'term': 'RAG',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Very common AI term; closely related to the course's glossary-search agent but not a glossary entry."},
           {'group': 'absent',
            'question': 'Can you explain RLHF?',
            'term': 'RLHF',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Training-related acronym that sits next to SFT/fine-tuning, so the model may borrow those definitions.'},
           {'group': 'absent',
            'question': 'What is a vector database used for?',
            'term': 'vector database',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Infrastructure term with no lexical overlap with glossary keys.'},
           {'group': 'absent',
            'question': 'What is an embedding in machine learning?',
            'term': 'embedding',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Generic ML term the base model knows well; tests abstaining despite confident knowledge.'},
           {'group': 'absent',
            'question': 'How does beam search decoding work?',
            'term': 'beam search',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Contains the word 'search' (the course builds a glossary search) but is a decoding method."},
           {'group': 'absent',
            'question': 'What is knowledge distillation?',
            'term': 'knowledge distillation',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': 'Training technique adjacent to fine-tuning/LoRA.'},
           {'group': 'absent',
            'question': 'What is a Mixture of Experts model?',
            'term': 'Mixture of Experts',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Architecture term; 'model' is generic and not a glossary key."},
           {'group': 'absent',
            'question': 'What is prompt injection?',
            'term': 'prompt injection',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Deliberately contains the glossary term 'prompt': a substring-based search will wrongly assert the Prompt definition."},
           {'group': 'absent',
            'question': 'What does top-p sampling do?',
            'term': 'top-p',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "Generation setting adjacent to the glossary's 'temperature' entry."},
           {'group': 'absent',
            'question': 'What is chain-of-thought prompting?',
            'term': 'chain-of-thought',
            'gold': 'That term is not in the course glossary.',
            'must_include': [],
            'why': "'prompting' is close to 'prompt'; tests the fuzzy-match threshold."}]}

In [ ]:
import hashlib, re

# 1) Training data = script output, verified against the recorded sha256
sft_text = "\n".join(SFT_JSONL_LINES) + "\n"
assert hashlib.sha256(sft_text.encode("utf-8")).hexdigest() == SFT_SHA256, "Embedded JSONL differs from the script output"
plain_records = [json.loads(line) for line in SFT_JSONL_LINES]

# Cross-check: regenerate with the same templates as scripts/make_sft_dataset.py from the embedded glossary
TERMS = GLOSSARY["terms"]
TEMPLATES = ["What is {term}?", "Briefly explain {term}."]   # Copied from scripts/make_sft_dataset.py

def qa(question: str, answer: str) -> dict:
    """One record in the same messages format as the script (and the original notebook's to_chat, without a system prompt)"""
    return {"messages": [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]}

assert plain_records == [qa(tpl.format(term=t["term"]), t["definition"]) for t in TERMS for tpl in TEMPLATES]
print(f"sha256 OK | plain records={len(plain_records)} (= {len(TERMS)} terms x {len(TEMPLATES)} templates)")

# 2) Augmented variant: alias questions + abstention examples for training-only absent terms
ABSTAIN_ANSWER = EVAL_SET["abstain_answer"]          # "That term is not in the course glossary."
TRAIN_ABSENT_TERMS = EVAL_SET["train_absent_terms"]  # 8 terms, disjoint from the evaluation's absent terms
alias_records = [qa(f"What does {alias} mean?", t["definition"])
                 for t in TERMS for alias in t["aliases"]
                 if alias.lower() != t["term"].lower() and len(alias) > 1]   # Skip aliases equal to the term and the 1-letter alias "r"
abstain_records = [qa(tpl.format(term=x), ABSTAIN_ANSWER) for x in TRAIN_ABSENT_TERMS for tpl in TEMPLATES]
train_records = plain_records if VARIANT == "plain" else plain_records + alias_records + abstain_records

# 3) Checks
def norm(text: str) -> str:
    """Lowercase, turn punctuation into spaces, collapse whitespace (also used by the glossary search in section 6)"""
    return " ".join(re.sub(r"[^0-9a-z]+", " ", text.lower()).split())

def glossary_keys(t: dict) -> list:
    """Term, its parts around parentheses ('LLM (Large Language Model)' -> 'LLM', 'Large Language Model'), and aliases"""
    head, _, inner = t["term"].partition("(")
    parts = [t["term"], head] + inner.rstrip(")").split("/") + t["aliases"]
    return [p.strip() for p in parts if p.strip()]

GLOSSARY_KEYS = {norm(k) for t in TERMS for k in glossary_keys(t)}
EVAL_ITEMS = EVAL_SET["items"]
absent_terms = [it["term"] for it in EVAL_ITEMS if it["group"] == "absent"] + TRAIN_ABSENT_TERMS
for x in absent_terms:   # The absent term must not be (part of) any glossary term or alias
    clash = [k for k in GLOSSARY_KEYS if f" {norm(x)} " in f" {k} "]
    assert not clash, f"'{x}' appears in glossary keys {clash}"
shared_words = {x: [k for k in GLOSSARY_KEYS if f" {k} " in f" {norm(x)} "] for x in absent_terms}
print("Absent terms containing a glossary word (intentional search traps):", {x: k for x, k in shared_words.items() if k})
assert not set(map(norm, TRAIN_ABSENT_TERMS)) & set(map(norm, absent_terms[:-len(TRAIN_ABSENT_TERMS)])), "Train/eval absent terms overlap"

train_questions = {r["messages"][0]["content"] for r in plain_records + alias_records + abstain_records}
unseen = [it for it in EVAL_ITEMS if it["group"] != "in_glossary"]
overlap = [it["question"] for it in unseen if it["question"] in train_questions]
assert not overlap, f"Evaluation questions found in training data: {overlap}"
assert all(it["question"] in {r["messages"][0]["content"] for r in plain_records} for it in EVAL_ITEMS if it["group"] == "in_glossary")
assert len({r["messages"][0]["content"] for r in train_records}) == len(train_records), "Duplicate training questions"

groups = {g: sum(it["group"] == g for it in EVAL_ITEMS) for g in ("in_glossary", "paraphrased", "absent")}
print(f"variant={VARIANT} | train={len(train_records)} (plain {len(plain_records)} + alias {len(alias_records)} + abstain {len(abstain_records)}"
      f"{'' if VARIANT == 'augmented' else ', extras not used'}) | eval={len(EVAL_ITEMS)} {groups} | overlap=0 | absent terms checked={len(absent_terms)}")
print("One training example:", json.dumps(train_records[0], ensure_ascii=False)[:300])
_ = (WORK / f"train_{VARIANT}.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in train_records), encoding="utf-8")
_ = (WORK / "eval_set.json").write_text(json.dumps(EVAL_SET, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Load the Model and Evaluate Before Training

**What to Do**: Load the base model without quantization (32-bit, unchanged), ask the 35 evaluation questions, and score the answers.

**Change from the original**: questions are sent as a **plain user message**, exactly as in the training data (no FAQ system prompt, no JSON format). `generate()` is otherwise unchanged (`enable_thinking=False`, greedy decoding).

**Metrics** (the same functions score section 5 and the glossary search in section 6)

| Metric | Groups | Meaning |
| --- | --- | --- |
| `keyword_hit` | in_glossary, paraphrased | All `must_include` keywords appear (case-insensitive, whole-word) |
| `keyword_frac` | in_glossary, paraphrased | Fraction of `must_include` keywords present |
| `f1` | in_glossary, paraphrased | Token-level F1 between the answer and the gold definition (SQuAD-style; rewards matching the glossary wording) |
| `abstained` | all | The answer says the term is not in the glossary / it does not know (regex list `ABSTAIN_PATTERNS`). For `absent` this is the desired behavior; otherwise the answer is `asserted`. For glossary groups it counts **false abstentions** |
| `ok` | all | `keyword_hit` for glossary groups, `abstained` for `absent` (used for regressions) |

**What to Look For**: in the original Japanese exercise style, before training the model typically gives fluent **general-knowledge** definitions: decent F1 for general terms, but misses course-specific keywords (`gemini-3.5-flash-lite`, `Lecture 02`) and asserts definitions for absent terms rather than abstaining. Check this in your table.

The harmless warnings described in the original notebook (unauthenticated HF Hub requests, linear-attention fallback) also appear here.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# No quantization. Keep weights in 32-bit (specify dtype because omitting it loads the distributed precision)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| Weight memory (GB):", BASE_MEMORY_GB)

# Change from the original: no FAQ system prompt. Ask the question as a plain user message, the same format as the training data
def build_messages(question: str):
    return [{"role": "user", "content": question}]

def generate_chat(model, messages: list):
    """Unchanged generation code, taking a message list (also used by the search + model condition in section 6b)"""
    # enable_thinking=False: leave the thinking block empty and generate only the answer (explicit for consistent conditions across models)
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def generate(model, question: str):
    return generate_chat(model, build_messages(question))

In [ ]:
from collections import Counter
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

GROUPS = ["in_glossary", "paraphrased", "absent"]
# "Not in the glossary / I don't know" phrasings counted as abstention
ABSTAIN_PATTERNS = [
    r"\bnot (?:in|included in|listed in|found in|part of|covered (?:in|by)|defined in|present in|mentioned in) (?:the |this |our )?(?:course(?:'s)? )?glossary",
    r"\bglossary (?:does not|doesn't|did not) (?:contain|include|list|cover|define|mention)",
    r"\b(?:no|not an?) (?:entry|definition|information) (?:for|on|about|in)\b",
    r"\b(?:i )?(?:do not|don't|cannot|can't) (?:know|find|answer)\b",
    r"\b(?:not sure|not familiar|unable to (?:find|answer)|no information)\b",
]

def is_abstention(text: str) -> bool:
    return any(re.search(p, text.lower()) for p in ABSTAIN_PATTERNS)

def tokens(text: str) -> list:
    return re.findall(r"[0-9a-z]+", text.lower())

def token_f1(pred: str, gold: str) -> float:
    """SQuAD-style token F1 (bag of lowercase alphanumeric tokens)"""
    p, g = tokens(pred), tokens(gold)
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0: return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

def has_keyword(text: str, kw: str) -> bool:
    """Case-insensitive whole-word match (so 'PR' would not match 'pretrained')"""
    return re.search(r"(?<!\w)" + re.escape(kw.lower()) + r"(?!\w)", text.lower()) is not None

def score(item: dict, output: str) -> dict:
    abstained = is_abstention(output)
    row = {"group": item["group"], "question": item["question"], "expected": item.get("term_id") or "(absent)", "output": output, "abstained": abstained}
    if item["group"] == "absent":
        row.update(keyword_hit=None, keyword_frac=None, f1=None, verdict="abstained" if abstained else "asserted", ok=abstained)
    else:
        hits = [has_keyword(output, k) for k in item["must_include"]]
        row.update(keyword_hit=all(hits), keyword_frac=sum(hits) / len(hits), f1=round(token_f1(output, item["gold"]), 3),
                   verdict="abstained" if abstained else ("hit" if all(hits) else "partial" if any(hits) else "miss"), ok=all(hits))
    return row

def summarize(rows: list, label: str, seconds: float) -> dict:
    """Per-group summary. Glossary groups: keyword hit rate, keyword fraction, F1, false-abstention rate. Absent: abstain vs assert rate"""
    out = {"label": label, "seconds": round(seconds, 1)}
    for g in GROUPS:
        rs = [r for r in rows if r["group"] == g]
        s = {"n": len(rs), "abstain_rate": round(sum(r["abstained"] for r in rs) / len(rs), 3)}
        if g == "absent":
            s["assert_rate"] = round(1 - s["abstain_rate"], 3)
        else:
            s.update(keyword_hit_rate=round(sum(r["keyword_hit"] for r in rs) / len(rs), 3),
                     keyword_frac=round(sum(r["keyword_frac"] for r in rs) / len(rs), 3),
                     f1=round(sum(r["f1"] for r in rs) / len(rs), 3))
        out[g] = s
    return out

def summary_table(*summaries) -> pd.DataFrame:
    """One row per (label, group) for side-by-side display"""
    return pd.DataFrame([{"label": s["label"], "group": g, **s[g]} for s in summaries for g in GROUPS]).set_index(["group", "label"]).sort_index(level=0, sort_remaining=False)

In [ ]:
def evaluate(model, label: str):
    """Generate and score all evaluation questions; save outputs in eval_<label>.json"""
    rows, t0 = [], time.time()
    for item in EVAL_ITEMS:
        rows.append(score(item, generate(model, item["question"])))
    summary = summarize(rows, label, time.time() - t0)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(summary_table(before_summary))
display(pd.DataFrame(before_rows)[["group", "question", "expected", "output", "verdict", "keyword_frac", "f1"]])

## 4. LoRA Training (Original Japanese Run: 4–5 Minutes on a T4)

### LoRA versus QLoRA

- **LoRA**: Freeze the base-model weights and train only small matrices (adapters) added to some layers.
- **QLoRA**: Keep base-model weights quantized to 4-bit and train only adapters in the same way. The adapters themselves are not quantized.

QLoRA reduces the memory used to hold base-model weights. It uses about a quarter of the memory of 16-bit, so models too large to fit on a GPU otherwise can be trained (for 7 billion parameters, about 14GB at 16-bit versus about 3.5GB at 4-bit).

| Aspect | LoRA (no quantization) | QLoRA (4-bit) |
| --- | --- | --- |
| Weight memory | Large | Small (about 1/4 of 16-bit) |
| Base-model outputs | Unchanged | May change due to quantization error; the size of the effect depends on the model |
| Training speed | Baseline | Not faster. Often slower because 4-bit weights must be converted back for each computation |
| Dependencies | peft | peft plus bitsandbytes (requires a CUDA GPU) |

Notes for implementing QLoRA:

- Pass `BitsAndBytesConfig(load_in_4bit=True, ...)` when loading, then run `prepare_model_for_kbit_training(model)` before adding LoRA.
- Storage precision (4-bit) and compute precision (`bnb_4bit_compute_dtype`, such as 16-bit) are separate settings.
- Compare before and after training using the same quantization conditions for the base model. Measuring without quantization before training and with 4-bit afterward mixes quantization and training effects.
- Adapters are created for the base model used in training. Record the base-model ID, revision, and quantization settings together when distributing or reusing them.
- 4-bit training is discouraged for some models. One [training-library guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune) discourages 4-bit training of Qwen3.5 because its quantization error is larger than usual.

This exercise's model fits on a T4 without quantization, so we train with LoRA. Consider LoRA first if the model fits; if it does not, consider QLoRA or a smaller model. You can examine memory and output differences when loading in 4-bit in section 8 (Advanced).

### Run Training

**What to Do**: Train LoRA on `train_records` (116 records for plain, 217 for augmented) for `EPOCHS` epochs, save the adapter in `adapter/` and the full training log in `loss_{RUN_NAME}.json`.

**Change from the original**: only the data. The records are already in the `messages` format that `to_chat` produced in the original notebook (here without a system prompt, matching the evaluation prompt), so they go directly into `Dataset.from_list`. There is **no validation split**: memorizing every glossary entry is the goal, so no glossary record is held out; the evaluation questions are kept out of training instead (checked in section 2). LoRA settings, batch 4 × accumulation 2, cosine schedule, warmup, `logging_steps=10`, checkpoints every 10 steps, and `fp16=False` are unchanged.

**What to Look For**

- `trainable%`: unchanged from the original exercise (same LoRA configuration).
- `Training Loss` every 10 steps: it should fall steadily (plain × 20 epochs = 300 steps, augmented × 14 = 392). A loss near 0 means the definitions are memorized almost verbatim.
- `Training time: ... | Steps: 300 | Epochs: 20.0` (plain, `EPOCHS = 20`) or `Steps: 392 | Epochs: 14.0` (augmented, `EPOCHS = 14`).
- The loss table printed after training (also saved in `loss_{RUN_NAME}.json`). There is no eval loss: every glossary record is trained, so there is no held-out split.

**Harmless Warning**: `The tokenizer has new PAD/BOS/EOS tokens ...` (same as the original).

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Change from the original: the glossary records are already in messages format (user question -> assistant definition)
# and match build_messages() used for evaluation, so pass them directly. No validation split (see the explanation above)
train_ds = Dataset.from_list(train_records)

model.config.use_cache = False
# Qwen3.5 uses linear attention in 18 of 24 layers; only the remaining 6 have q/k/v/o_proj. All layers have gate/up/down_proj (MLP)
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Normally leave unchanged. True enables 16-bit mixed-precision computation (CUDA only; weights stay 32-bit). Faster, but loss may become nan
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Fraction of parameters trained with LoRA
t0 = time.time()
trainer.train()  # See 4b to resume if interrupted
print(f"Training time: {time.time()-t0:.0f} seconds | Steps: {trainer.state.global_step} | Epochs: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # Training mode enables dropout and makes evaluation outputs vary (assignment suppresses displaying the whole model)

In [ ]:
# Save the full training log (every logging step) right away, so it survives a later failure
LOG_HISTORY = trainer.state.log_history
train_loss_log = [{k: e.get(k) for k in ("step", "epoch", "loss", "learning_rate", "grad_norm")} for e in LOG_HISTORY if "loss" in e]
eval_loss_log = [{k: e.get(k) for k in ("step", "epoch", "eval_loss")} for e in LOG_HISTORY if "eval_loss" in e]   # Empty: no eval split
loss_record = {"run_name": RUN_NAME, "variant": VARIANT, "epochs": EPOCHS, "logging_steps": trainer.args.logging_steps,
               "global_step": trainer.state.global_step, "train_loss": train_loss_log, "eval_loss": eval_loss_log,
               "eval_loss_note": "No eval split: every glossary record is trained, so the trainer does not evaluate",
               "log_history": LOG_HISTORY}
(WORK / f"loss_{RUN_NAME}.json").write_text(json.dumps(loss_record, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(train_loss_log))

### 4b. If Training Stops or Cannot Run

**Stopped midway (runtime disconnected, etc.)**: Checkpoints are saved every 10 steps in `/content/qlora_lab/ckpt/`. If the files remain, rerun sections 0–3 with the same `VARIANT` and `EPOCHS`, then change `trainer.train()` to `trainer.train(resume_from_checkpoint=True)` and run it.

**Could not train (no GPU available or not enough time)**: sections 5, 6b and 8 need the GPU model, but section 6 (glossary search) runs on CPU. Set `executed_training_myself` in section 7 to `False` and state clearly how far you ran the notebook yourself.

## 5. Evaluate After Training (Same Questions and Generation Settings as Before)

**What to Do**: Evaluate the trained model `tuned` on exactly the same 35 questions with the same generation settings. The only change is whether adapters are present.

**What to Look For**

- Comparison table: how did `keyword_hit_rate` and `f1` change for `in_glossary` (seen questions) versus `paraphrased` (unseen wording)? How did `abstain_rate` change for `absent`?
- `Regressions`: questions that were `ok` before training but not after (for example, an absent term that the base model declined to define but the tuned model now defines).
- Failure types in the side-by-side table:
  - **Wrong entry**: a paraphrase answered with the definition of a neighboring term (for example, LoRA instead of rank).
  - **Asserted fabrication**: an absent term answered in glossary style ("...In this course, ...") — training teaches the *style* of course definitions, not knowledge of what the glossary contains.
  - **False abstention** (augmented variant): a glossary term answered with "not in the course glossary".

In [ ]:
after_summary, after_rows = evaluate(tuned, f"after_{RUN_NAME}")   # -> eval_after_{RUN_NAME}.json
display(summary_table(before_summary, after_summary))

# Questions that were ok before training (keyword hit / abstained) but not after
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
improved = [b["question"] for b, a in zip(before_rows, after_rows) if not b["ok"] and a["ok"]]
print(f"Regressions ({len(regressed)}):", regressed)
print(f"Improved ({len(improved)}):", improved)

detail = pd.DataFrame([{"Group": b["group"], "Question": b["question"], "Expected": b["expected"],
                        "Before output": b["output"], "Before": b["verdict"], "Before F1": b["f1"],
                        "After output": a["output"], "After": a["verdict"], "After F1": a["f1"]}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. Compare with Glossary Search

**Replaces the original section 6.** The original asked knowledge questions to show that learning a format does not guarantee knowledge. Here the question is the homework's: **is training the model the right way to give it the glossary, compared with searching the glossary** (the mechanism you build as a tool in Lecture 03)?

`glossary_search(question)` is deterministic and uses no model:

1. **Exact match**: normalize the question and every glossary key (term, the parts of the term inside and outside parentheses, aliases) to lowercase words without punctuation, and find keys that occur as whole words. 1-character keys (`r`) are ignored. If several match, prefer the longest key, with a small bonus when the glossary spelling appears with the same capitalization (acronyms such as `HITL`), so that `HITL` beats the generic alias `agent`.
2. **Fuzzy fallback** (only if nothing matched exactly): compare every 1–4 word span of the question with keys of ≥ 4 characters using `difflib.SequenceMatcher`; accept the best if its ratio ≥ `FUZZY_THRESHOLD` (0.88, high enough that `prompting` does not match `prompt`).
3. Return that entry's definition verbatim, or `"That term is not in the course glossary."` if nothing matched.

It is scored with the **same metrics on the same evaluation set**, plus latency per question and whether the retrieved entry is the expected one.

**What to Look For**: search returns the glossary text verbatim, so when it finds the right entry, F1 is 1.0 and keywords always hit. Its failures are different from the model's: questions that describe a concept without naming it (no match → false abstention), generic words matching the wrong entry (`prompt injection` → `Prompt`), and choosing the wrong entry among several matches.

In [ ]:
import difflib

FUZZY_THRESHOLD = 0.88
STOPWORDS = set("a an the is are was what which who how why when where do does did i my me we our you your of in on for to and or with by from this that it its be can about mean means explain briefly tell".split())

# Index: (normalized key, original spelling, term) for every term, term part, and alias
INDEX = [(norm(k), k, t) for t in TERMS for k in glossary_keys(t) if len(norm(k)) > 1]

def glossary_search(question: str):
    """Return (answer, matched term id or None, how it matched)"""
    q = f" {norm(question)} "
    exact = [(len(key) + (3 if any(ch.isupper() for ch in orig) and re.search(rf"(?<!\w){re.escape(orig)}(?!\w)", question) else 0), key, t)
             for key, orig, t in INDEX if f" {key} " in q]
    if exact:
        _, key, t = max(exact, key=lambda x: x[0])
        return t["definition"], t["id"], f"exact:{key}"
    words = q.split()
    spans = {" ".join(words[i:j]) for i in range(len(words)) for j in range(i + 1, min(i + 4, len(words)) + 1)}
    spans = [s for s in spans if len(s) >= 4 and not set(s.split()) <= STOPWORDS]
    best = max(((difflib.SequenceMatcher(None, s, key).ratio(), s, key, t) for s in spans for key, _, t in INDEX if len(key) >= 4),
               key=lambda x: x[0], default=None)
    if best and best[0] >= FUZZY_THRESHOLD:
        ratio, span, key, t = best
        return t["definition"], t["id"], f"fuzzy:{span}~{key}({ratio:.2f})"
    return ABSTAIN_ANSWER, None, "none"

def evaluate_search(label: str = "search"):
    rows, latencies = [], []
    for item in EVAL_ITEMS:
        t0 = time.perf_counter()
        answer, term_id, how = glossary_search(item["question"])
        latencies.append(time.perf_counter() - t0)
        rows.append({**score(item, answer), "retrieved": term_id, "match": how,
                     "retrieved_correct": term_id == item.get("term_id")})   # absent: correct only if nothing was retrieved
    summary = summarize(rows, label, sum(latencies))
    summary["avg_latency_ms"] = round(1000 * sum(latencies) / len(latencies), 3)
    summary["retrieval_accuracy"] = {g: round(sum(r["retrieved_correct"] for r in rows if r["group"] == g) / sum(r["group"] == g for r in rows), 3) for g in GROUPS}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary, rows

In [ ]:
search_summary, search_rows = evaluate_search()
print("Average search latency (ms):", search_summary["avg_latency_ms"], "| retrieval accuracy:", search_summary["retrieval_accuracy"])
display(pd.DataFrame(search_rows)[["group", "question", "expected", "retrieved", "match", "verdict", "f1"]])

# Before training / after training / search, side by side (search needs no GPU, so it also runs if training was skipped)
summaries = [s for s in (globals().get("before_summary"), globals().get("after_summary"), search_summary) if s]
display(summary_table(*summaries))
if "after_rows" in globals():
    display(pd.DataFrame([{"Group": a["group"], "Question": a["question"], "Tuned model": a["verdict"], "Tuned F1": a["f1"],
                           "Search": s["verdict"], "Search F1": s["f1"], "Search match": s["match"]}
                          for a, s in zip(after_rows, search_rows)]))

## 6b. Search + Model (Retrieve, Then Generate)

The two conditions above are extremes: the model answers from its weights only, or the search returns text with no model. Here they are combined (the pattern used with a glossary tool in later lectures):

1. Run the same `glossary_search(question)`.
2. **Entry found** → the model gets one user message containing that entry's term and definition and the question, with the instruction to answer **only from the entry**.
3. **Nothing found** → the fixed abstention `"That term is not in the course glossary."` is returned **without calling the model**.

It runs with the **base model** (adapter disabled with `tuned.disable_adapter()`; this is the important comparison: does retrieval alone give what training gives?) and with the **trained model**, on the same 35 questions with the same greedy settings and metrics. About 1.5 minutes per model (abstained questions skip generation). Outputs: `eval_rag_base.json`, `eval_rag_trained_{RUN_NAME}.json`.

**What to Look For**: retrieval errors propagate (a wrong entry gives a wrong answer; no match gives a false abstention), so differences from `search` show what the model adds or loses: does it paraphrase away keywords (lower F1 / keyword hit than verbatim search), does it follow "answer only from the entry", and does the trained model copy the definition more faithfully?

In [ ]:
RAG_INSTRUCTION = ("Answer the question using only the course glossary entry below. "
                   "Do not add information that is not in the entry.")

def build_rag_messages(question: str, entry: dict) -> list:
    """One user message: instruction + retrieved glossary entry (term and definition) + question"""
    return [{"role": "user", "content": f"{RAG_INSTRUCTION}\n\nGlossary entry:\nTerm: {entry['term']}\nDefinition: {entry['definition']}\n\nQuestion: {question}"}]

TERM_BY_ID = {t["id"]: t for t in TERMS}

def rag_answer(question: str, gen_fn):
    """gen_fn(messages) -> str. Returns (answer, retrieved id or None, how matched, whether the model was called)"""
    _, term_id, how = glossary_search(question)
    if term_id is None:
        return ABSTAIN_ANSWER, None, how, False          # Fixed abstention, no model call
    return gen_fn(build_rag_messages(question, TERM_BY_ID[term_id])), term_id, how, True

def evaluate_rag(gen_fn, label: str):
    rows, t0 = [], time.time()
    for item in EVAL_ITEMS:
        answer, term_id, how, called = rag_answer(item["question"], gen_fn)
        rows.append({**score(item, answer), "retrieved": term_id, "match": how, "model_called": called,
                     "retrieved_correct": term_id == item.get("term_id")})
    summary = summarize(rows, label, time.time() - t0)
    summary["model_calls"] = sum(r["model_called"] for r in rows)
    summary["retrieval_accuracy"] = {g: round(sum(r["retrieved_correct"] for r in rows if r["group"] == g) / sum(r["group"] == g for r in rows), 3) for g in GROUPS}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary, rows

In [ ]:
# Base model = trained PeftModel with the adapter switched off (training injected LoRA layers into `model` in place)
with tuned.disable_adapter():
    rag_base_summary, rag_base_rows = evaluate_rag(lambda m: generate_chat(tuned, m), "rag_base")                    # -> eval_rag_base.json
rag_trained_summary, rag_trained_rows = evaluate_rag(lambda m: generate_chat(tuned, m), f"rag_trained_{RUN_NAME}")   # -> eval_rag_trained_{RUN_NAME}.json
print("model calls:", rag_base_summary["model_calls"], "| seconds base / trained:", rag_base_summary["seconds"], "/", rag_trained_summary["seconds"])
display(summary_table(before_summary, after_summary, search_summary, rag_base_summary, rag_trained_summary))
display(pd.DataFrame([{"Group": s["group"], "Question": s["question"], "Retrieved": s["retrieved"], "Search": s["verdict"],
                       "RAG base": b["verdict"], "RAG base F1": b["f1"], "RAG base output": b["output"],
                       "RAG trained": t["verdict"], "RAG trained F1": t["f1"]}
                      for s, b, t in zip(search_rows, rag_base_rows, rag_trained_rows)]))

### Discussion: Training the Model versus Searching the Glossary

Fill this in with your own numbers from the tables above (and from both variants if you ran them), including the **search + model** rows. Points to address:

| Aspect | LoRA-trained model | Glossary search |
| --- | --- | --- |
| Seen questions (`in_glossary`) | Does it reproduce the definitions verbatim after `EPOCHS` epochs? | Verbatim by construction |
| Paraphrases / aliases | Can generalize to new wording and descriptions without the term name — check `paraphrased` | Only if a term/alias string (or a close spelling) appears; fails on descriptions, can pick the wrong entry |
| Absent terms | Plain training teaches the *style* of definitions, not which terms exist; does it assert? Did augmented abstention examples transfer to *unseen* absent terms? | Abstains whenever nothing matches; asserts only on accidental word overlap |
| Updating the glossary | Retrain (minutes of GPU) and re-evaluate for every change | Edit the JSON; effective immediately |
| Traceability | No source; cannot tell a memorized definition from a fabricated one | Returns the exact entry (citable) |
| Cost / latency | GPU, ~seconds per answer | Microseconds on CPU (`avg_latency_ms`) |

A common conclusion is that **knowledge** that changes and must be exact (a glossary) belongs in search/retrieval (Lecture 03's tool), while fine-tuning suits **format and behavior** (answer style, when to call the tool, abstaining). Check whether your results support that, and where they do not.

## 8. Advanced: Load with 4-bit Quantization and Compare Memory and Outputs (Optional)

**What to Do**: Load the same model in 4-bit (NF4) and observe changes in weight memory and before-training outputs. QLoRA adds LoRA to this 4-bit base model and trains it. No training is done here.

- bitsandbytes requires a CUDA GPU, so this runs only in a GPU runtime. Installation takes about 1 minute.
- 4-bit loading places the model on the GPU with `device_map="auto"` (matching bitsandbytes' loading method).
- If you also try training the 4-bit model, run `prepare_model_for_kbit_training(model_4bit)` first, follow section 4's training procedure, and evaluate both before and after using a 4-bit base model.

**What to Look For**

- Weight memory: compare 32-bit and 4-bit values.
- Comparison table: since no training occurred, differences from the 32-bit before-training results (`before`) reflect quantization alone (keyword hits, F1, and abstentions on the same 35 questions).

**What is compared**: as in the original, the **base model** (no adapter) loaded in 4-bit NF4 versus the same base model in 32-bit (`before`), on the same 35 glossary questions with the same greedy settings and metrics. The trained adapter is **not** put on the 4-bit base: the original section 8 compares base weights only, and an adapter trained on 32-bit weights evaluated on a quantized base would mix two effects (that combination is what QLoRA *trains*, which is out of scope here). No training happens in this section.

**Changes from the original**: this notebook's evaluation and metrics; it runs **before** section 7 so its summary goes into `result_{RUN_NAME}.json` (key `4bit`) and `eval_4bit_{RUN_NAME}.json` is downloaded with the other files. `RUN_4BIT = True` by default (about 6–8 extra minutes: ~1 min bitsandbytes install, ~0.5 min load, ~4–6 min generation). Set it to `False` to skip. Before loading, cached GPU memory is released (`gc.collect()` + `torch.cuda.empty_cache()`); the 32-bit model (~3.2 GB) and the 4-bit model (<1 GB) fit together on a 15 GB T4, and the 4-bit model is deleted afterwards.

In [ ]:
RUN_4BIT = True   # Optional section 8 observation (no training). False skips it (~6-8 minutes saved)
q_summary = None
if not RUN_4BIT:
    print("Section 8 skipped (RUN_4BIT = False)")
else:
    import subprocess, sys, gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()   # Release cached activations/optimizer buffers from training before loading a second model
        print(f"GPU memory allocated before 4-bit load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.50.*"], check=True)
    from transformers import BitsAndBytesConfig

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Storage precision: 4-bit (NF4)
                             bnb_4bit_compute_dtype=torch.float16,             # Compute precision: 16-bit (T4 does not support bf16)
                             bnb_4bit_use_double_quant=True)
    model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
    print("Weight memory (GB) | No quantization (32-bit):", BASE_MEMORY_GB, "| 4-bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

    # No training occurred, so differences from the unquantized before-training results (before) reflect quantization
    q_summary, q_rows = evaluate(model_4bit, f"4bit_{RUN_NAME}")   # -> eval_4bit_{RUN_NAME}.json (base model, 4-bit)
    q_summary["weight_memory_gb"] = {"fp32": BASE_MEMORY_GB, "4bit": round(model_4bit.get_memory_footprint() / 1e9, 2)}
    q_summary["compared_with"] = "before (fp32 base model, no adapter)"
    q_summary["regressed_vs_fp32"] = [b["question"] for b, q in zip(before_rows, q_rows) if b["ok"] and not q["ok"]]
    q_summary["improved_vs_fp32"] = [b["question"] for b, q in zip(before_rows, q_rows) if not b["ok"] and q["ok"]]
    display(summary_table(before_summary, q_summary))
    display(pd.DataFrame([{"Group": b["group"], "Question": b["question"], "32-bit before": b["verdict"], "4-bit before": q["verdict"],
                           "4-bit output": q["output"]} for b, q in zip(before_rows, q_rows)]))
    del model_4bit; gc.collect(); torch.cuda.empty_cache()

## 7. Comparison Table and Submission

**What to Do**: Combine settings, variant, epochs, library versions, the per-group before / after / search / search + model summaries, regressions, training-set size, training time, and the loss curve into `result_{RUN_NAME}.json`.

Files per run (`RUN_NAME` = e.g. `plain_ep20`): `result_{RUN_NAME}.json`, `eval_before.json`, `eval_after_{RUN_NAME}.json`, `eval_search.json`, `eval_rag_base.json`, `eval_rag_trained_{RUN_NAME}.json`, `loss_{RUN_NAME}.json`, and `eval_4bit_{RUN_NAME}.json` (section 8, if `RUN_4BIT = True`). (`eval_before`, `eval_search`, `eval_rag_base` do not depend on training, so they are the same for every run.)

The last cell downloads exactly these files (Colab only). With `USE_DRIVE = True` they are also copied to Drive. If you could not train, set `executed_training_myself` to `False`.

In [ ]:
log = trainer.state.log_history
losses = [e["loss"] for e in log if "loss" in e]
rag_base = globals().get("rag_base_summary"); rag_trained = globals().get("rag_trained_summary")
train_end = next((e for e in reversed(log) if "train_runtime" in e), {})
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "variant": VARIANT,
    "epochs": EPOCHS,
    "run_name": RUN_NAME,
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (no quantization)",
    "train_set_size": len(train_records),
    "train_set_breakdown": {"plain": len(plain_records), "alias": len(alias_records) if VARIANT == "augmented" else 0,
                            "abstain": len(abstain_records) if VARIANT == "augmented" else 0},
    "sft_jsonl_sha256": SFT_SHA256,
    "train_runtime_seconds": train_end.get("train_runtime"),
    "train_steps": trainer.state.global_step,
    "final_loss": losses[-1] if losses else None,
    "train_loss_curve": [[e["step"], round(e["loss"], 4)] for e in log if "loss" in e],   # [step, loss] at every logging step
    "eval_loss_curve": [[e["step"], e["eval_loss"]] for e in log if "eval_loss" in e],      # Empty: no eval split
    "mean_train_loss": train_end.get("train_loss"),
    "eval_set_size": {g: sum(it["group"] == g for it in EVAL_ITEMS) for g in GROUPS},
    "before": before_summary, "after": after_summary, "search": search_summary,
    "rag_base": rag_base, "rag_trained": rag_trained,
    "4bit": globals().get("q_summary"),   # section 8: base model 4-bit vs fp32 (None if RUN_4BIT = False)   # search + model (section 6b)
    "regressed": regressed, "improved": improved,
    "executed_training_myself": True,   # Set False if you could not run training yourself
}
(WORK / f"result_{RUN_NAME}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
OUTPUT_FILES = [f"result_{RUN_NAME}.json", "eval_before.json", f"eval_after_{RUN_NAME}.json", "eval_search.json",
                "eval_rag_base.json", f"eval_rag_trained_{RUN_NAME}.json", f"loss_{RUN_NAME}.json"]
if globals().get("q_summary"):
    OUTPUT_FILES.append(f"eval_4bit_{RUN_NAME}.json")
print("Missing:", [n for n in OUTPUT_FILES if not (WORK / n).exists()] or "none")
if USE_DRIVE:
    import shutil
    for name in OUTPUT_FILES:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Saved to Drive:", CONFIG["drive_dir"])

In [ ]:
# Download the result files to your computer (Colab only; elsewhere, collect them from WORK)
try:
    from google.colab import files
    for name in OUTPUT_FILES:
        if (WORK / name).exists():
            files.download(str(WORK / name))
except Exception as e:   # Not in Colab, or the browser blocked multiple downloads
    print(f"Automatic download unavailable ({type(e).__name__}). Files are in {WORK}:", [n for n in OUTPUT_FILES if (WORK / n).exists()])